In [1]:
import kagglehub

# Download latest version
path_data = kagglehub.dataset_download("nyvantran6634/data-processed-v3", output_dir="./kaggle")
path_cache = kagglehub.dataset_download("nyvantran6634/cache-dataset-processed-v3", output_dir="./kaggle")

print("Path to path_data dataset files:", path_data)

print("Path to path_cache dataset files:", path_cache)

Using Colab cache for faster access to the 'data-processed-v3' dataset.


100%|██████████| 1.11G/1.11G [01:06<00:00, 17.8MB/s]

Extracting files...


Path to path_data dataset files: /kaggle/input/data-processed-v3
Path to path_cache dataset files: ./kaggle


In [11]:
# =====================================================================
# File: config.py
# Dự án: Driver Guardian - Module LiteFat (IEEE/RSJ IROS 2025 Upgraded)
# Tác vụ: Định nghĩa lớp cấu hình TrainConfig và tài liệu hướng dẫn
#         chi tiết bằng Tiếng Việt cho toàn bộ pipeline huấn luyện.
# =====================================================================
from __future__ import annotations

import json
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple


@dataclass
class TrainConfig:
    """
    Cấu hình toàn diện cho quá trình huấn luyện và kiểm định mô hình LiteFat.

    Hướng dẫn chọn chế độ huấn luyện:
    --------------------------------
    1. Chế độ Đa phương thức hoàn chỉnh (Multimodal Mode - Khuyến nghị):
       - Đặt: `use_visual_context = True`
       - Hoạt động: Kết hợp cả chuỗi tọa độ 68 điểm mốc mặt (động học IOD) và
         chuỗi ảnh crop khuôn mặt 224x224 qua MobileNetV3-Small.
       - Chiến lược tối ưu:
         + Đặt `freeze_backbone_epochs = 5` để đóng băng MobileNetV3 trong 5 epoch đầu,
           buộc nhánh Động học & Đồ thị STGL học hội tụ trước.
         + Sau đó mở khóa với `backbone_lr = 5e-5` (nhỏ hơn 10 lần so với `lr = 5e-4` của GCN).

    2. Chế độ Chỉ dùng Landmark siêu tốc (Fast Landmark-Only Mode):
       - Đặt: `use_visual_context = False`
       - Hoạt động: Tắt hoàn toàn nhánh MobileNetV3, bỏ qua việc đọc giải mã video .mp4.
         Dataset đọc trực tiếp 100% từ cache .pt trên RAM/NVMe.
       - Tốc độ: Huấn luyện cực nhanh (~10-15 giây / epoch), không ngốn VRAM GPU.
    """

    # =================================================================
    # 1. ĐƯỜNG DẪN DỮ LIỆU & BỘ DỮ LIỆU CON
    # =================================================================
    # Thư mục gốc chứa các video đã qua tiền xử lý (chứa dataset_merged_split.csv)
    dataset_dir: str = r"/kaggle/input/data-processed-v3"

    # Thư mục chứa các tệp cache đặc trưng .pt đã trích xuất sẵn (train.pt, val.pt, test.pt)
    cache_dir: str = r"./kaggle/LiteFat/cache_pt"

    # Cờ sử dụng cache .pt (True: nạp từ cache để tăng tốc; False: trích xuất on-the-fly)
    use_cache: bool = True

    # Danh sách các bộ dữ liệu con được sử dụng để huấn luyện và đánh giá.
    # Đặt `None` để sử dụng tất cả các bộ dữ liệu có trong manifest/cache.
    # Ví dụ: ["SUST-DDD", "UL-DD", "VBDDD"]
    selected_datasets: Optional[List[str]] = None

    # Tên file manifest CSV phân chia tập dữ liệu
    manifest_filename: str = "dataset_merged_split.csv"

    # Thư mục lưu kết quả huấn luyện (checkpoints, metrics json, cấu hình)
    checkpoint_dir: str = "checkpoints/LiteFat_v5"

    # Thư mục lưu log TensorBoard
    log_dir: str = "checkpoints/LiteFat_v5/logs"

    # =================================================================
    # 2. CHẾ ĐỘ ĐA PHƯƠNG THỨC & NGỮ CẢNH THỊ GIÁC (VISUAL CONTEXT)
    # =================================================================
    # Bật/tắt nhánh MobileNetV3-Small trích xuất ngữ cảnh thị giác từ ảnh khuôn mặt.
    # True: Huấn luyện đa phương thức (Landmark + Face Crops).
    # False: Chỉ huấn luyện đồ thị từ Landmark (không cần giải mã video, siêu nhanh).
    use_visual_context: bool = False

    # Kích thước ảnh khung hình đưa vào trích xuất điểm mốc letterbox (w, h)
    letterbox_size: Tuple[int, int] = (640, 640)

    # Kích thước ảnh khuôn mặt crop chuẩn hóa cho MobileNetV3-Small (w, h)
    face_size: Tuple[int, int] = (224, 224)

    # =================================================================
    # 3. LẤY MẪU THỜI GIAN (TEMPORAL SAMPLING)
    # =================================================================
    # Khoảng cách lấy mẫu thời gian giữa các khung hình (giây).
    # 0.05s tương ứng 20 FPS (khớp chuẩn cache .pt, ghi nhận đầy đủ chớp mắt 100-300ms).
    # 0.1s tương ứng 10 FPS.
    time_interval: float = 0.05

    # Độ dài cửa sổ trượt (giây).
    # 15.0s tương ứng T = 300 frames ở 20 FPS (ghi nhận toàn diện chuỗi hành vi ngủ gật, bảo toàn 83% video UL-DD).
    # Đặt None để lấy toàn bộ độ dài video.
    window_sec: Optional[float] = 12.0

    # =================================================================
    # 4. TĂNG CƯỜNG DỮ LIỆU ĐA PHƯƠNG THỨC (DATA AUGMENTATION)
    # =================================================================
    # Bật/tắt tăng cường dữ liệu trên tập Train
    augment_train: bool = True

    # Xác suất lật ngang đối xứng (Horizontal Flip) đồng thời cả Landmark 68 điểm và ảnh khuôn mặt
    flip_prob: float = 0.5

    # Biên độ co giãn tỉ lệ khuôn mặt ngẫu nhiên (Scale Jitter)
    scale_jitter: float = 0.08

    # Biên độ dịch chuyển tọa độ khuôn mặt ngẫu nhiên (Shift Jitter)
    shift_jitter: float = 0.05

    # Góc quay 2D nghiêng đầu ngẫu nhiên trong mặt phẳng (Roll Rotation, độ)
    roll_deg: float = 12.0

    # Tỷ lệ che khuất ngẫu nhiên vài khung hình mô phỏng mất tracking (Temporal Dropout)
    temporal_dropout: float = 0.05

    # Độ lệch chuẩn của nhiễu Gaussian cộng vào tọa độ điểm mốc (được làm mượt thời gian)
    landmark_noise_std: float = 0.0003

    # Biên độ biến đổi độ sáng và độ tương phản nhất quán thời gian trên ảnh khuôn mặt
    color_jitter: float = 0.2

    # Xác suất che khuất một phần vùng ảnh ngẫu nhiên (Spatial Erasing / Cutout)
    spatial_dropout: float = 0.15

    # =================================================================
    # 5. THAM SỐ KIẾN TRÚC MÔ HÌNH LITEFAT
    # =================================================================
    num_nodes: int = 68
    feature_dim: int = 64
    mobilenet_proj_hidden: int = 192
    node_embed_dim: int = 16
    residual_channels: int = 64
    dilation_channels: int = 64
    skip_channels: int = 128
    end_channels: int = 256
    num_classes: int = 2
    num_blocks: int = 3
    layers_per_block: int = 3
    kernel_size: int = 3
    dropout: float = 0.35
    pretrained_backbone: bool = True

    # Cặp chỉ số đỉnh giải phẫu dọc mắt (EAR) và dọc miệng (MAR) để tạo cạnh kề cơ sở A_prior
    # Cặp mắt trái: (37, 41), (38, 40); Cặp mắt phải: (43, 47), (44, 46)
    # Cặp miệng: (50, 58), (51, 57), (52, 56), (61, 67), (62, 66), (63, 65)
    vertical_pairs: List[Tuple[int, int]] = field(default_factory=lambda: [
        (37, 41), (38, 40), (43, 47), (44, 46),
        (50, 58), (51, 57), (52, 56), (61, 67), (62, 66), (63, 65)
    ])

    # Cặp chỉ số liên vùng (Mày <-> Mắt, Mũi <-> Mắt, Miệng <-> Cằm)
    cross_region_pairs: List[Tuple[int, int]] = field(default_factory=lambda: [
        (19, 37), (24, 44), (30, 39), (30, 42), (57, 8), (51, 33)
    ])

    # =================================================================
    # 6. THAM SỐ HUẤN LUYỆN & TỐI ƯU HÓA (OPTIMIZATION)
    # =================================================================
    epochs: int = 30
    batch_size: int = 16
    num_workers: int = 0
    pin_memory: bool = True
    seed: int = 42

    # Lấy mẫu cân bằng chủ thể (Subject-Balanced Sampling qua WeightedRandomSampler)
    subject_balanced_sampling: bool = True

    # Tốc độ học cơ bản cho nhánh Đồ thị STGL, Dung hợp FiLM và Đầu phân loại
    lr: float = 3e-4

    # Tốc độ học riêng biệt cho xương sống MobileNetV3-Small (thường nhỏ hơn 10 lần)
    backbone_lr: float = 3e-5

    # Hệ số phạt suy giảm trọng số (Weight Decay L2)
    weight_decay: float = 2e-3

    # Số epoch đầu tiên đóng băng hoàn toàn MobileNetV3-Small (chỉ huấn luyện STGL)
    freeze_backbone_epochs: int = 5

    # Số epoch làm nóng tốc độ học (Linear Warmup)
    warmup_epochs: int = 0

    # Giảm tốc độ học cho các trọng số đã được tiền huấn luyện (Warm-start) để chống quên thảm khốc
    pretrained_lr_scale: float = 0.25

    # Giới hạn chuẩn đạo hàm để chống nổ gradient (Gradient Clipping)
    grad_clip: float = 1.0

    # Sử dụng bộ nhớ hỗn hợp Automatic Mixed Precision (AMP float16)
    use_amp: bool = True

    # Lưu lại checkpoint sau mỗi save_freq epoch
    save_freq: int = 1

    # Đường dẫn tới tệp checkpoint để tiếp tục huấn luyện (ví dụ: "checkpoints/LiteFat/epoch_03.pt" hoặc "checkpoints/LiteFat/last.pt").
    resume: Optional[str] = "checkpoints/LiteFat_v5/last.pt"

    # Đường dẫn checkpoint tiền huấn luyện để khởi tạo ấm (None khi train lại từ đầu cho v5)
    pretrained_from: Optional[str] = None
    # =================================================================
    # 7. HÀM MẤT MÁT (LOSS CONFIGURATION)
    # =================================================================
    # Trọng số của hàm mất mát cấp đoạn video (Clip-level CrossEntropy)
    clip_loss_weight: float = 1.0

    # Trọng số của hàm mất mát từng khung hình có trọng số thời gian nhân quả (Causal Frame Loss)
    frame_loss_weight: float = 0.3

    # Hệ số làm mượt nhãn (Label Smoothing) chống overfitting
    label_smoothing: float = 0.1

    def to_dict(self) -> Dict[str, Any]:
        """Chuyển cấu hình thành từ điển (Dict)."""
        return asdict(self)

    def save_json(self, path: str | Path) -> None:
        """Lưu cấu hình ra file JSON phục vụ tái lập thực nghiệm."""
        p = Path(path)
        p.parent.mkdir(parents=True, exist_ok=True)
        with open(p, "w", encoding="utf-8") as f:
            json.dump(self.to_dict(), f, indent=2, ensure_ascii=False)

    @classmethod
    def from_json(cls, path: str | Path) -> TrainConfig:
        """Nạp cấu hình từ file JSON."""
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        import dataclasses
        valid_fields = {f.name for f in dataclasses.fields(cls)}
        filtered_data = {k: v for k, v in data.items() if k in valid_fields}
        return cls(**filtered_data)


In [3]:
# =====================================================================
# File: ultis.py
# Dự án: Driver Guardian - Module LiteFat
# Tác vụ: Thư viện tiện ích dùng chung: xử lý ảnh, cắt vùng khuôn mặt,
#         tăng cường dữ liệu đa phương thức đối xứng 68 điểm mốc,
#         thiết lập hạt giống tái lập, logging và lưu trữ checkpoint.
# =====================================================================
from __future__ import annotations

import logging
import math
import os
import random
import sys
import tempfile
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

import cv2
import numpy as np
import torch
import torch.nn.functional as F

# Bảng hoán vị đối xứng trái - phải cho 68 điểm mốc khuôn mặt chuẩn 300-W
FLIP_MAP_68 = [
    # 0..16: Đường viền hàm dưới (17 điểm) hoán vị đối xứng
    16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 0,
    # 17..26: Lông mày trái (17..21) <-> Lông mày phải (22..26)
    26, 25, 24, 23, 22, 21, 20, 19, 18, 17,
    # 27..35: Sống mũi (27..30 đối xứng chính nó) & Cánh mũi (31..35)
    27, 28, 29, 30, 35, 34, 33, 32, 31,
    # 36..47: Mắt trái (36..41) <-> Mắt phải (42..47)
    45, 44, 43, 42, 47, 46, 39, 38, 37, 36, 41, 40,
    # 48..67: Viền môi ngoài (48..59) & Viền môi trong (60..67)
    54, 53, 52, 51, 50, 49, 48, 59, 58, 57, 56, 55,
    64, 63, 62, 61, 60, 67, 66, 65,
]


def _parse_size(size: Optional[Union[int, Tuple[int, int], List[int]]]) -> Tuple[int, int]:
    """
    Chuẩn hóa kích thước về tuple (width, height).
    """
    if size is None:
        return (640, 640)
    if isinstance(size, int):
        return (size, size)
    if isinstance(size, (tuple, list)) and len(size) == 2:
        return (int(size[0]), int(size[1]))
    raise ValueError(f"Kích thước không hợp lệ: {size}. Phải là int hoặc tuple (w, h).")


def letterbox(
        image: np.ndarray,
        new_size: Optional[int | Tuple[int, int] | List[int]] = (640, 640),
        color: Tuple[int, int, int] = (114, 114, 114),
) -> np.ndarray:
    """
    Resize ảnh giữ nguyên tỷ lệ khung hình gốc với đệm màu viền về kích thước đích (w, h).
    - image: Ảnh BGR numpy array shape (H, W, 3).
    - new_size: Kích thước đích theo (w, h) (vd: (640, 480)) hoặc số nguyên S tương ứng (S, S).
    - color: Màu đệm viền BGR, mặc định (114, 114, 114).
    Trả về ảnh numpy array có shape (target_h, target_w, 3).
    """
    target_w, target_h = _parse_size(new_size)
    if target_w <= 0 or target_h <= 0:
        return image

    h, w = image.shape[:2]
    if w == target_w and h == target_h:
        return image

    scale = min(target_w / float(w), target_h / float(h))
    nw, nh = max(1, round(w * scale)), max(1, round(h * scale))
    interp = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_LINEAR
    resized = cv2.resize(image, (nw, nh), interpolation=interp)

    if nw == target_w and nh == target_h:
        return resized

    canvas = np.full((target_h, target_w, 3), color, dtype=image.dtype)
    x, y = (target_w - nw) // 2, (target_h - nh) // 2
    canvas[y:y + nh, x:x + nw] = resized
    return canvas


def compute_iou(box_a: np.ndarray, box_b: np.ndarray) -> float:
    """
    Tính Intersection over Union (IoU) giữa 2 hộp giới hạn [x1, y1, x2, y2].
    """
    xa = max(box_a[0], box_b[0])
    ya = max(box_a[1], box_b[1])
    xb = min(box_a[2], box_b[2])
    yb = min(box_a[3], box_b[3])

    inter = max(0.0, xb - xa) * max(0.0, yb - ya)
    area_a = max(0.0, box_a[2] - box_a[0]) * max(0.0, box_a[3] - box_a[1])
    area_b = max(0.0, box_b[2] - box_b[0]) * max(0.0, box_b[3] - box_b[1])
    union = area_a + area_b - inter
    if union <= 0.0:
        return 0.0
    return float(inter / union)


def crop_faces_from_boxes(
        frames: List[np.ndarray],
        crop_boxes: Union[np.ndarray, torch.Tensor],
        face_size: Tuple[int, int] = (224, 224),
) -> torch.Tensor:
    """
    Cắt vùng khuôn mặt chuẩn hóa 224x224 cho nhánh MobileNetV3-Small từ crop_boxes:
      - Nhận vào danh sách frames BGR (đã qua letterbox) và crop_boxes [T, 4] dạng [x1, y1, x2, y2].
      - Resize về face_size (w, h), chuyển sang RGB [0, 1] và chuẩn hóa ImageNet mean/std.
      - Trả về torch.Tensor [T, 3, crop_h, crop_w].
    """
    T = len(frames)
    crop_w, crop_h = _parse_size(face_size)
    if T == 0:
        return torch.empty((0, 3, crop_h, crop_w), dtype=torch.float32)

    if isinstance(crop_boxes, torch.Tensor):
        crop_boxes = crop_boxes.detach().cpu().numpy()
    crop_boxes = np.asarray(crop_boxes, dtype=np.int32)

    face_crops = np.zeros((T, crop_h, crop_w, 3), dtype=np.uint8)

    for t in range(T):
        if t < len(crop_boxes):
            x1, y1, x2, y2 = crop_boxes[t]
        else:
            x1, y1, x2, y2 = 0, 0, frames[t].shape[1], frames[t].shape[0]

        crop = frames[t][y1:y2, x1:x2, :]
        if crop.size == 0 or crop.shape[0] < 2 or crop.shape[1] < 2:
            crop = frames[t]

        resized = cv2.resize(crop, (crop_w, crop_h), interpolation=cv2.INTER_LINEAR)
        face_crops[t] = resized[:, :, ::-1]  # Chuyển BGR sang RGB

    # Chuẩn hóa ImageNet vector hóa cho toàn bộ chuỗi T ảnh
    face_tensor = torch.from_numpy(face_crops).permute(0, 3, 1, 2).float() / 255.0
    norm_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    norm_std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    face_tensor = (face_tensor - norm_mean) / norm_std
    return face_tensor


def augment_multimodal_clip(
        face_frames: Optional[torch.Tensor],
        landmarks: torch.Tensor,
        is_train: bool = True,
        flip_prob: float = 0.5,
        scale_jitter: float = 0.08,
        shift_jitter: float = 0.05,
        roll_deg: float = 12.0,
        temporal_dropout: float = 0.05,
        landmark_noise_std: float = 0.002,
        color_jitter: float = 0.2,
        spatial_dropout: float = 0.15,
) -> Tuple[Optional[torch.Tensor], torch.Tensor]:
    """
    Tăng cường dữ liệu không gian - thời gian đồng bộ cho cả chuỗi điểm mốc và ảnh mặt:
    1. Lật ngang đối xứng (Horizontal Flip) đồng bộ cả mốc và ảnh:
       - Tọa độ x -> (1.0 - x).
       - Hoán vị đỉnh giải phẫu 68 điểm theo FLIP_MAP_68 (mắt trái <-> mắt phải).
       - Lật trục ngang chiều rộng của face_frames.
    2. Quay 2D trong mặt phẳng (In-plane Roll Rotation) góc ngẫu nhiên quanh tâm mặt.
    3. Co giãn bất đẳng hướng (Anisotropic Scale: scale_x != scale_y) và dịch chuyển (Shift Jitter).
    4. Thêm nhiễu Gaussian nhẹ (Landmark Noise) trên tọa độ điểm mốc hợp lệ.
    5. Che khuất tạm thời vài khung hình (Temporal Frame Dropout) mô phỏng mất tracking.
    6. Biến đổi độ sáng/độ tương phản (Color Jitter) nhất quán thời gian trên face_frames.
    7. Cắt ngẫu nhiên vùng ảnh (Random Spatial Cutout / Erasing) trên face_frames.
    """
    if not is_train:
        return face_frames, landmarks

    lms = landmarks.clone()
    frames = face_frames.clone() if face_frames is not None else None

    # Xác định các điểm hợp lệ (không phải đệm 1.0)
    valid_pts = (lms < 0.999).all(dim=-1, keepdim=True)  # [T, 68, 1]

    # 1. Lật ngang đối xứng (Horizontal Flip)
    if random.random() < flip_prob:
        # Lật tọa độ x của điểm mốc
        lms[..., 0] = torch.where(valid_pts.squeeze(-1), 1.0 - lms[..., 0], lms[..., 0])
        # Hoán vị đỉnh giải phẫu đối xứng chuẩn 300W
        flip_idx = torch.as_tensor(FLIP_MAP_68, dtype=torch.long, device=lms.device)
        lms = lms[:, flip_idx, :]

        # Lật ngang chiều rộng của ảnh khuôn mặt
        if frames is not None:
            frames = torch.flip(frames, dims=[-1])

    # 2. Quay 2D góc nghiêng đầu (In-plane Roll Rotation quanh tâm mặt)
    if roll_deg > 0 and random.random() < 0.5:
        angle_deg = random.uniform(-roll_deg, roll_deg)
        theta = math.radians(angle_deg)
        cos_t = math.cos(theta)
        sin_t = math.sin(theta)

        valid_coords = lms[valid_pts.squeeze(-1)]
        if valid_coords.numel() > 0:
            center_x = valid_coords[:, 0].mean().item()
            center_y = valid_coords[:, 1].mean().item()

            dx = lms[..., 0] - center_x
            dy = lms[..., 1] - center_y
            rot_x = dx * cos_t - dy * sin_t + center_x
            rot_y = dx * sin_t + dy * cos_t + center_y

            rot_x = torch.clamp(rot_x, 0.0, 1.0)
            rot_y = torch.clamp(rot_y, 0.0, 1.0)

            lms[..., 0] = torch.where(valid_pts.squeeze(-1), rot_x, lms[..., 0])
            lms[..., 1] = torch.where(valid_pts.squeeze(-1), rot_y, lms[..., 1])

    # 3. Co giãn gần như đẳng hướng (Near-Isotropic Scale Jitter) và dịch chuyển (Shift)
    # Giới hạn biến dạng tỷ lệ trục y/x trong ngưỡng ±1.5% để bảo toàn độ mở mắt EAR và miệng MAR
    if scale_jitter > 0 or shift_jitter > 0:
        base_scale = 1.0 + random.uniform(-scale_jitter, scale_jitter)
        aspect_ratio = 1.0 + random.uniform(-0.015, 0.015)
        scale_x = base_scale
        scale_y = base_scale * aspect_ratio
        shift_x = random.uniform(-shift_jitter, shift_jitter)
        shift_y = random.uniform(-shift_jitter, shift_jitter)

        # Tính tâm trung bình của các điểm hợp lệ
        valid_coords = lms[valid_pts.squeeze(-1)]
        if valid_coords.numel() > 0:
            center_x = valid_coords[:, 0].mean().item()
            center_y = valid_coords[:, 1].mean().item()

            new_x = (lms[..., 0] - center_x) * scale_x + center_x + shift_x
            new_y = (lms[..., 1] - center_y) * scale_y + center_y + shift_y

            new_x = torch.clamp(new_x, 0.0, 1.0)
            new_y = torch.clamp(new_y, 0.0, 1.0)

            lms[..., 0] = torch.where(valid_pts.squeeze(-1), new_x, lms[..., 0])
            lms[..., 1] = torch.where(valid_pts.squeeze(-1), new_y, lms[..., 1])

    # 4. Nhiễu Gaussian mượt theo thời gian (Temporally Smoothed Landmark Noise)
    # Bộ lọc 3-tap [0.25, 0.50, 0.25] tránh tạo ra gai đạo hàm bậc 2 (acc) giả,
    # và giữ nguyên tọa độ gốc của 2 khóe mắt (36, 45) để bảo vệ mẫu số IOD
    if landmark_noise_std > 0:
        raw_noise = torch.randn_like(lms) * landmark_noise_std
        if lms.shape[0] >= 3:
            smooth_noise = 0.5 * raw_noise
            smooth_noise[1:-1] += 0.25 * (raw_noise[:-2] + raw_noise[2:])
            smooth_noise[0] += 0.25 * raw_noise[0]
            smooth_noise[-1] += 0.25 * raw_noise[-1]
            noise = smooth_noise
        else:
            noise = raw_noise

        # Không làm nhiễu 2 nút gốc đo khoảng cách mắt (36, 45)
        noise[:, 36, :] = 0.0
        noise[:, 45, :] = 0.0

        noisy_lms = torch.clamp(lms + noise, 0.0, 1.0)
        lms = torch.where(valid_pts, noisy_lms, lms)

    # 5. Temporal Frame Dropout: che khuất ngẫu nhiên 1-3 khung hình mô phỏng mất tracking
    if temporal_dropout > 0 and random.random() < 0.3:
        T_len = lms.shape[0]
        num_drop = min(max(1, int(round(T_len * temporal_dropout))), 4)
        if T_len > num_drop:
            start_drop = random.randint(0, T_len - num_drop)
            lms[start_drop:start_drop + num_drop] = 1.0
            if frames is not None:
                frames[start_drop:start_drop + num_drop] = 0.0

    # 6. Tăng cường trên khung hình khuôn mặt (nhất quán thời gian)
    if frames is not None and frames.numel() > 0:
        # Biến đổi độ sáng & tương phản nhất quán cho toàn bộ T frames
        if color_jitter > 0 and random.random() < 0.5:
            brightness_factor = 1.0 + random.uniform(-color_jitter, color_jitter)
            contrast_factor = 1.0 + random.uniform(-color_jitter, color_jitter)
            mean_val = frames.mean(dim=(-2, -1), keepdim=True)
            frames = (frames - mean_val) * contrast_factor + mean_val
            frames = frames * brightness_factor

        # Spatial Dropout / Cutout: Che khuất một vùng nhỏ ngẫu nhiên trên mặt
        if spatial_dropout > 0 and random.random() < spatial_dropout:
            _, _, H, W = frames.shape
            cut_h = int(H * random.uniform(0.1, 0.25))
            cut_w = int(W * random.uniform(0.1, 0.25))
            y1 = random.randint(0, max(0, H - cut_h))
            x1 = random.randint(0, max(0, W - cut_w))
            frames[:, :, y1:y1 + cut_h, x1:x1 + cut_w] = 0.0

    return frames, lms


def seed_everything(seed: int = 42) -> None:
    """Thiết lập hạt giống ngẫu nhiên để tái lập kết quả thực nghiệm."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def configure_logging(output_dir: Union[str, Path]) -> logging.Logger:
    """Cấu hình logging đồng thời ra Console và File log."""
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    log_file = out_path / "train.log"

    logger = logging.getLogger("DriverGuardian.LiteFat")
    logger.setLevel(logging.INFO)
    logger.propagate = False

    # Xóa các handler cũ nếu có
    for h in list(logger.handlers):
        logger.removeHandler(h)

    formatter = logging.Formatter(
        fmt="[%(asctime)s] [%(levelname)s] %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    console_h = logging.StreamHandler(sys.stdout)
    console_h.setLevel(logging.INFO)
    console_h.setFormatter(formatter)
    logger.addHandler(console_h)

    file_h = logging.FileHandler(str(log_file), mode="a", encoding="utf-8")
    file_h.setLevel(logging.INFO)
    file_h.setFormatter(formatter)
    logger.addHandler(file_h)

    return logger


def atomic_save(obj: Any, path: Union[str, Path]) -> None:
    """Lưu tệp an toàn (Atomic Save) chống lỗi tệp hỏng khi bị ngắt đột ngột."""
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    tmp_file = tempfile.NamedTemporaryFile(delete=False, dir=str(p.parent), suffix=".tmp")
    tmp_path = Path(tmp_file.name)
    try:
        tmp_file.close()
        torch.save(obj, tmp_path)
        if p.exists():
            p.unlink()
        tmp_path.rename(p)
    except Exception as e:
        if tmp_path.exists():
            tmp_path.unlink()
        raise e


def compute_metrics(labels: np.ndarray, preds: np.ndarray, probs: Optional[np.ndarray] = None) -> Dict[str, float]:
    """
    Tính toán các chỉ số đánh giá phân loại buồn ngủ:
    Accuracy, Precision, Recall, Specificity, F1-Score, F2-Score và ROC-AUC (nếu có probs).
    """
    labels = np.asarray(labels, dtype=int).ravel()
    preds = np.asarray(preds, dtype=int).ravel()

    tp = int(np.sum((labels == 1) & (preds == 1)))
    fp = int(np.sum((labels == 0) & (preds == 1)))
    fn = int(np.sum((labels == 1) & (preds == 0)))
    tn = int(np.sum((labels == 0) & (preds == 0)))
    total = len(labels)

    acc = (tp + tn) / max(1, total)
    prec = tp / max(1, tp + fp)
    rec = tp / max(1, tp + fn)
    spec = tn / max(1, tn + fp)
    f1 = (2.0 * prec * rec) / max(1e-8, prec + rec) if (prec + rec) > 0 else 0.0
    f2 = (5.0 * prec * rec) / max(1e-8, 4.0 * prec + rec) if (4.0 * prec + rec) > 0 else 0.0

    metrics = {
        "accuracy": float(acc),
        "precision": float(prec),
        "recall": float(rec),
        "specificity": float(spec),
        "f1": float(f1),
        "f2": float(f2),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }

    if probs is not None:
        try:
            from sklearn.metrics import roc_auc_score
            probs_arr = np.asarray(probs, dtype=float).ravel()
            if len(np.unique(labels)) > 1:
                metrics["auc"] = float(roc_auc_score(labels, probs_arr))
            else:
                metrics["auc"] = 0.5
        except Exception:
            metrics["auc"] = 0.5

    return metrics


In [4]:
# =====================================================================
# File: dataset.py
# Dự án: Driver Guardian - Module LiteFat
# Tác vụ: Nạp tập dữ liệu phát hiện buồn ngủ, trích xuất 68 điểm mốc khuôn mặt
#         bằng PIPNet (FaceBoxesV2 + ResNet18 300W) và cắt vùng khuôn mặt
#         chuẩn hóa cho mô hình LiteFat (MobileNetV3-Small + STGL).
# Tối ưu hóa:
#   - Tốc độ: Batched GPU FaceBoxesV2, Cached Priors, Torchvision GPU NMS,
#             Batched GPU PIPNet với giải mã NRM vector hóa, Fast Video Seeking.
#   - Chính xác: Đồng bộ kích thước letterbox (ultis.py), theo dõi khuôn mặt tài xế
#               qua IoU, Landmark-Guided BBox Refinement, nội suy tuyến tính &
#               làm mượt thời gian (Temporal Smoothing) cho khung hình khuôn mặt.
# =====================================================================
from __future__ import annotations

import csv
import json
import logging
import math
import os
import random
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

# Monkey patch cho scipy >= 1.14 để tương thích với torchlm metrics
import scipy.integrate

if not hasattr(scipy.integrate, "simps"):
    scipy.integrate.simps = scipy.integrate.simpson

# Cấu hình môi trường Windows và nạp DLL PyTorch CUDA
torch_lib_path = os.path.join(os.path.dirname(sys.executable), "Library", "bin")
if os.path.exists(torch_lib_path):
    os.environ["PATH"] = torch_lib_path + os.pathsep + os.environ.get("PATH", "")
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

if hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader, Dataset

# Import hàm chuẩn hóa và tăng cường từ ultis.py (đã được định nghĩa ở Cell trước)
try:
    from ultis import letterbox, _parse_size, compute_iou, crop_faces_from_boxes, augment_multimodal_clip
except ImportError:
    pass

# Khởi tạo mô hình PIPNet 68 điểm mốc trên GPU khi cần trích xuất trực tiếp on-the-fly
try:
    from pipnet_68_camera import init_pipnet_68_gpu
except ImportError:
    def init_pipnet_68_gpu(backbone: str = "resnet18"):
        """Khởi tạo FaceBoxesV2 + PIPNet (68 điểm 300W) trên GPU thông qua torchlm."""
        import torchlm
        from torchlm.tools import faceboxesv2
        from torchlm.models import pipnet

        dev = "cuda" if torch.cuda.is_available() else "cpu"
        logger.info(f"Đang khởi tạo PIPNet ({backbone} - 68 điểm 300W) trên thiết bị: {dev.upper()}")
        torchlm.runtime.bind(faceboxesv2(device=dev))
        torchlm.runtime.bind(
            pipnet(
                backbone=backbone,
                pretrained=True,
                num_nb=10,
                num_lms=68,
                net_stride=32,
                input_size=256,
                meanface_type="300w",
                map_location=dev,
            )
        )
        logger.info("Đã nạp xong trọng số mô hình PIPNet 68!")

logger = logging.getLogger("DriverGuardian.LiteFat")
if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("[%(levelname)s] %(name)s: %(message)s"))
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)


# Các hàm _parse_size, compute_iou, crop_faces_from_boxes đã được tập trung trong ultis.py.


# =====================================================================
# 1. BỘ TRÍCH XUẤT PIPNET TỐI ƯU HÓA CAO CẤP (FAST PIPNET EXTRACTOR)
# =====================================================================
class FastPIPNetExtractor:
    """
    Bộ trích xuất 68 điểm mốc khuôn mặt và cắt khung hình cho LiteFat
    được tối ưu hóa tối đa:
      1. Batched FaceBoxesV2 trên GPU: Nạp một mảng T khung hình cùng lúc,
         tái sử dụng PriorBox anchors đã được tính toán sẵn 1 lần trên GPU,
         sử dụng torchvision.ops.nms trực tiếp trên GPU.
      2. Temporal Driver Tracking: Lựa chọn khuôn mặt tài xế ổn định dựa trên
         điểm số, diện tích và độ tương đồng IoU qua các khung hình liên tiếp.
      3. Batched PIPNet 300W trên GPU: Gom toàn bộ vùng mặt phát hiện được thành
         1 batch tensor [N_valid, 3, 256, 256], chạy 1 lần suy luận duy nhất,
         giải mã NRM (Neighbor Regression) hoàn toàn bằng tensor PyTorch trên GPU.
      4. Landmark-Guided Box Refinement: Mở rộng hộp bao kết hợp giữa BBox và
         cực trị 68 điểm mốc, làm mượt theo thời gian (EMA) và nội suy tuyến tính
         cho các khung hình mất mặt, đảm bảo không bị cắt lẹm cằm/mắt.
    """

    def __init__(
            self,
            backbone: str = "resnet18",
            device: Optional[str] = None,
            letterbox_size: Tuple[int, int] = (640, 640),
            face_size: Tuple[int, int] = (224, 224),
            conf_threshold: float = 0.5,
            nms_threshold: float = 0.3,
            bbox_margin: float = 0.15,
            box_smooth_alpha: float = 0.7,
            use_fp16: bool = False,
    ):
        self.device = torch.device(
            device if device is not None else ("cuda" if torch.cuda.is_available() else "cpu")
        )
        self.letterbox_size = _parse_size(letterbox_size)  # (target_w, target_h)
        self.face_size = _parse_size(face_size)  # (crop_w, crop_h)
        self.conf_threshold = float(conf_threshold)
        self.nms_threshold = float(nms_threshold)
        self.bbox_margin = float(bbox_margin)
        self.box_smooth_alpha = float(box_smooth_alpha)
        self.use_fp16 = bool(use_fp16) and (self.device.type == "cuda")

        # Khởi tạo mô hình thông qua runtime của torchlm
        from torchlm.runtime._wrappers import RuntimeWrapper
        if getattr(RuntimeWrapper, "face_base", None) is None or getattr(RuntimeWrapper, "landmarks_base",
                                                                         None) is None:
            init_pipnet_68_gpu(backbone=backbone)

        self.face_base = RuntimeWrapper.face_base
        self.face_net = self.face_base.net.to(self.device)
        self.face_net.eval()

        self.lmk_base = RuntimeWrapper.landmarks_base
        self.lmk_net = self.lmk_base.to(self.device)
        self.lmk_net.eval()

        # Cache PriorBox và hằng số chuẩn hóa trên GPU cho kích thước letterbox cố định
        target_w, target_h = self.letterbox_size
        from torchlm.tools._faceboxesv2 import PriorBox
        pbox = PriorBox(self.face_base.cfg, image_size=(target_h, target_w))
        self.cached_priors = pbox.forward().to(self.device)  # [P, 4]
        self.cached_variances = self.face_base.cfg["variance"]
        self.cached_scale = torch.tensor(
            [target_w, target_h, target_w, target_h], dtype=torch.float32, device=self.device
        )
        self.faceboxes_mean = torch.tensor(
            [104.0, 117.0, 123.0], dtype=torch.float32, device=self.device
        ).view(1, 3, 1, 1)

        # Hằng số chuẩn hóa PIPNet (ImageNet RGB)
        self.pipnet_mean = torch.tensor(
            [0.485, 0.456, 0.406], dtype=torch.float32, device=self.device
        ).view(1, 3, 1, 1)
        self.pipnet_std = torch.tensor(
            [0.229, 0.224, 0.225], dtype=torch.float32, device=self.device
        ).view(1, 3, 1, 1)

        # Cache bảng chỉ số NRM cho PIPNet trên GPU
        rev1 = self.lmk_base.reverse_index1
        rev2 = self.lmk_base.reverse_index2
        num_nb = self.lmk_base.num_nb
        self.rev_indices = torch.tensor(
            [r1 * num_nb + r2 for r1, r2 in zip(rev1, rev2)],
            dtype=torch.long,
            device=self.device,
        )

        # Hằng số chuẩn hóa ImageNet cho MobileNetV3 (CPU / GPU)
        self.imagenet_mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 1, 3)
        self.imagenet_std = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 1, 3)

    @torch.no_grad()
    def _detect_faces_batched(self, frames: List[np.ndarray], max_chunk: int = 32) -> List[np.ndarray]:
        """
        Chạy phát hiện khuôn mặt theo lô T khung hình trên GPU.
        Tự động chia nhỏ theo max_chunk để tránh tràn VRAM GPU khi chuỗi khung hình dài (ví dụ video 300-1000 frames).
        Mỗi khung hình trong `frames` đã được chuẩn hóa kích thước letterbox (target_h, target_w).
        Trả về danh sách mảng numpy shape [num_dets, 5] (x1, y1, x2, y2, score).
        """
        T = len(frames)
        if T == 0:
            return []

        all_dets: List[np.ndarray] = []
        priors_xy = self.cached_priors[None, :, :2]
        priors_wh = self.cached_priors[None, :, 2:]
        v0, v1 = self.cached_variances[0], self.cached_variances[1]

        for s in range(0, T, max_chunk):
            chunk = frames[s:s + max_chunk]
            cur_T = len(chunk)

            # Gom batch BGR numpy -> tensor [cur_T, 3, H, W] trên GPU
            frames_np = np.stack(chunk, axis=0)  # [cur_T, H, W, 3]
            tensor = torch.from_numpy(frames_np).permute(0, 3, 1, 2).to(self.device).float()
            tensor = tensor - self.faceboxes_mean

            loc, conf = self.face_net(tensor)  # loc: [cur_T, P, 4], conf: [cur_T, P, 2]
            scores = conf[:, :, 1]  # [cur_T, P]

            # Giải mã tọa độ hộp giới hạn vector hóa cho cả chunk
            boxes_xy = priors_xy + loc[:, :, :2] * v0 * priors_wh
            boxes_wh = priors_wh * torch.exp(loc[:, :, 2:] * v1)
            b_x1y1 = boxes_xy - boxes_wh / 2.0
            b_x2y2 = b_x1y1 + boxes_wh
            decoded_boxes = torch.cat([b_x1y1, b_x2y2], dim=-1) * self.cached_scale  # [cur_T, P, 4]

            for b in range(cur_T):
                sc = scores[b]
                mask = sc >= self.conf_threshold
                if not mask.any():
                    all_dets.append(np.empty((0, 5), dtype=np.float32))
                    continue

                bx = decoded_boxes[b, mask]
                sc_valid = sc[mask]

                keep = torchvision.ops.nms(bx, sc_valid, iou_threshold=self.nms_threshold)
                keep = keep[:50]  # Giữ tối đa 50 mặt có độ tin cậy cao nhất
                res = torch.cat([bx[keep], sc_valid[keep, None]], dim=-1).cpu().numpy()
                all_dets.append(res)

        return all_dets

    def _select_driver_faces(
            self,
            all_dets: List[np.ndarray],
            target_w: int,
            target_h: int
    ) -> Tuple[List[Optional[np.ndarray]], List[bool]]:
        """
        Theo dõi và chọn duy nhất 1 khuôn mặt tài xế cho từng khung hình:
          - Kết hợp diện tích và độ tin cậy.
          - Duy trì tính nhất quán vị trí không gian qua IoU với khung hình liền kề.
        Trả về:
          - chosen_bboxes: danh sách bbox [x1, y1, x2, y2, score] (hoặc None nếu mất mặt)
          - found: danh sách boolean đánh dấu phát hiện được khuôn mặt
        """
        T = len(all_dets)
        chosen_bboxes: List[Optional[np.ndarray]] = [None] * T
        found: List[bool] = [False] * T

        prev_box: Optional[np.ndarray] = None
        prev_box_area = 1.0

        for t in range(T):
            dets = all_dets[t]
            if len(dets) == 0:
                continue

            best_idx = 0
            if prev_box is None:
                # Frame đầu tiên có mặt: chọn theo score * area
                best_score = -1.0
                for i, d in enumerate(dets):
                    area = max(0.0, d[2] - d[0]) * max(0.0, d[3] - d[1])
                    metric = d[4] * (area ** 0.5)
                    if metric > best_score:
                        best_score = metric
                        best_idx = i
            else:
                # Các frame tiếp theo: ưu tiên IoU với khuôn mặt đã khóa trước đó
                best_match = -1.0
                for i, d in enumerate(dets):
                    iou = compute_iou(d[:4], prev_box[:4])
                    area = max(0.0, d[2] - d[0]) * max(0.0, d[3] - d[1])
                    # Kết hợp IoU và độ tin cậy
                    metric = (iou * 2.0) + (d[4] * 0.5) + (min(1.0, area / (prev_box_area + 1e-6)) * 0.3)
                    if metric > best_match:
                        best_match = metric
                        best_idx = i

            chosen = dets[best_idx].copy()
            chosen_bboxes[t] = chosen
            found[t] = True
            prev_box = chosen
            prev_box_area = max(1.0, (prev_box[2] - prev_box[0]) * (prev_box[3] - prev_box[1]))

        return chosen_bboxes, found

    @torch.no_grad()
    def _extract_landmarks_batched(
            self,
            frames: List[np.ndarray],
            chosen_bboxes: List[Optional[np.ndarray]],
            found: List[bool],
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Trích xuất 68 điểm mốc khuôn mặt theo chuẩn 300W bằng PIPNet theo lô trên GPU.
        Đảm bảo giữ chính xác công thức căn chỉnh vùng cắt của PIPNet 300W:
          extend = 0.2 -> x1 -= 0.1*w, y1 += 0.1*h, x2 += 0.1*w, y2 += 0.1*h
        Trả về:
          - landmarks_norm: [T, 68, 2] gồm (x/W, y/H)
          - landmarks_px:   [T, 68, 2] tọa độ pixel tuyệt đối trên khung hình letterbox
        """
        T = len(frames)
        target_w, target_h = self.letterbox_size
        extend = 0.2

        valid_indices = [t for t in range(T) if found[t] and chosen_bboxes[t] is not None]

        # Khởi tạo ma trận điểm mốc: mặc định gán 1.0 khi mất mặt (theo Section III-B bài báo LiteFat)
        landmarks_norm = np.ones((T, 68, 2), dtype=np.float32)
        landmarks_px = np.zeros((T, 68, 2), dtype=np.float32)

        if not valid_indices:
            return landmarks_norm, landmarks_px

        crop_coords = []
        crops_resized = []

        for idx in valid_indices:
            bbox = chosen_bboxes[idx]
            bx1, by1, bx2, by2 = bbox[:4]
            bw = bx2 - bx1 + 1.0
            bh = by2 - by1 + 1.0

            # Căn chỉnh lề theo đúng logic của PIPNet 300W (torchlm RuntimeWrapper)
            cx1 = int(bx1 - bw * extend / 2.0)
            cy1 = int(by1 + bh * extend / 2.0)
            cx2 = int(bx2 + bw * extend / 2.0)
            cy2 = int(by2 + bh * extend / 2.0)

            cx1 = max(0, min(target_w - 2, cx1))
            cy1 = max(0, min(target_h - 2, cy1))
            cx2 = max(cx1 + 1, min(target_w, cx2))
            cy2 = max(cy1 + 1, min(target_h, cy2))

            crop_w = float(cx2 - cx1)
            crop_h = float(cy2 - cy1)
            crop_coords.append((cx1, cy1, crop_w, crop_h))

            # Cắt ảnh và chuyển BGR -> RGB cho PIPNet
            face_crop = frames[idx][cy1:cy2, cx1:cx2, ::-1]
            face_resized = cv2.resize(face_crop, (256, 256), interpolation=cv2.INTER_LINEAR)
            crops_resized.append(face_resized)

        # Gom lô N_valid vùng mặt lên GPU và giải mã theo từng sub-batch (max_chunk=64) để tránh tràn VRAM
        max_crop_chunk = 64
        batched_lms_list = []
        norm_factor = 1.0 * self.lmk_base.input_size / self.lmk_base.net_stride  # 256 / 32 = 8.0
        num_nb = self.lmk_base.num_nb
        max_len = self.lmk_base.max_len

        for s in range(0, len(crops_resized), max_crop_chunk):
            chunk_crops = crops_resized[s:s + max_crop_chunk]
            crops_np = np.stack(chunk_crops, axis=0)  # [cur_b, 256, 256, 3]
            crops_tensor = torch.from_numpy(crops_np).permute(0, 3, 1, 2).to(self.device).float() / 255.0
            crops_tensor = (crops_tensor - self.pipnet_mean) / self.pipnet_std

            out_cls, out_x, out_y, out_nb_x, out_nb_y = self.lmk_net(crops_tensor)
            cur_B, num_lms, H_map, W_map = out_cls.shape

            out_cls = out_cls.view(cur_B, num_lms, -1)
            max_ids = torch.argmax(out_cls, dim=2)
            max_ids_col = max_ids.unsqueeze(-1)

            out_x_flat = out_x.view(cur_B, num_lms, -1)
            out_y_flat = out_y.view(cur_B, num_lms, -1)
            x_sel = torch.gather(out_x_flat, 2, max_ids_col).squeeze(2)
            y_sel = torch.gather(out_y_flat, 2, max_ids_col).squeeze(2)

            max_ids_nb = max_ids_col.repeat(1, 1, num_nb).view(cur_B, num_lms * num_nb, 1)
            out_nb_x_flat = out_nb_x.view(cur_B, num_lms * num_nb, -1)
            out_nb_y_flat = out_nb_y.view(cur_B, num_lms * num_nb, -1)
            nb_x_sel = torch.gather(out_nb_x_flat, 2, max_ids_nb).view(cur_B, num_lms, num_nb)
            nb_y_sel = torch.gather(out_nb_y_flat, 2, max_ids_nb).view(cur_B, num_lms, num_nb)

            pred_x = ((max_ids % W_map).float() + x_sel) / norm_factor
            pred_y = (torch.floor(max_ids.float() / W_map) + y_sel) / norm_factor

            pred_nb_x = ((max_ids % W_map).float().unsqueeze(-1) + nb_x_sel) / norm_factor
            pred_nb_y = (torch.floor(max_ids.float() / W_map).unsqueeze(-1) + nb_y_sel) / norm_factor

            # Giải mã NRM vector hóa qua bảng ánh xạ reverse indices
            pred_nb_x_flat = pred_nb_x.view(cur_B, -1)
            pred_nb_y_flat = pred_nb_y.view(cur_B, -1)

            nb_x_rev = pred_nb_x_flat[:, self.rev_indices].view(cur_B, num_lms, max_len)
            nb_y_rev = pred_nb_y_flat[:, self.rev_indices].view(cur_B, num_lms, max_len)

            final_x = torch.mean(torch.cat([pred_x.unsqueeze(-1), nb_x_rev], dim=-1), dim=-1)
            final_y = torch.mean(torch.cat([pred_y.unsqueeze(-1), nb_y_rev], dim=-1), dim=-1)
            chunk_lms = torch.stack([final_x, final_y], dim=-1).cpu().numpy()  # [cur_B, 68, 2] in [0, 1]
            batched_lms_list.append(chunk_lms)

        batched_lms = np.concatenate(batched_lms_list, axis=0) if batched_lms_list else np.empty((0, 68, 2),
                                                                                                 dtype=np.float32)

        # Ánh xạ ngược tọa độ từ crop về khung hình letterbox
        for i, idx in enumerate(valid_indices):
            cx1, cy1, cw, ch = crop_coords[i]
            score = float(chosen_bboxes[idx][4])

            lm_crop = batched_lms[i].copy()
            abs_x = lm_crop[:, 0] * cw + cx1
            abs_y = lm_crop[:, 1] * ch + cy1

            landmarks_px[idx, :, 0] = abs_x
            landmarks_px[idx, :, 1] = abs_y

            # Chuẩn hóa về [0, 1] trên kích thước letterbox (chỉ lấy tọa độ x, y)
            landmarks_norm[idx, :, 0] = abs_x / float(target_w)
            landmarks_norm[idx, :, 1] = abs_y / float(target_h)

        return landmarks_norm, landmarks_px

    def compute_face_crop_boxes(
            self,
            chosen_bboxes: List[Optional[np.ndarray]],
            landmarks_px: np.ndarray,
            found: List[bool],
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Tính toán tọa độ điểm cắt khuôn mặt vuông cho từng khung hình:
          1. Kết hợp BBox và cực trị 68 điểm mốc (Landmark-Guided BBox Refinement).
          2. Mở rộng lề margin 15% đối xứng, giữ tỷ lệ khung hình vuông (1:1).
          3. Nội suy tuyến tính (Linear Interpolation) cho các frame mất mặt.
          4. Làm mượt thời gian bằng bộ lọc EMA (Exponential Moving Average).
          5. Bounding và ép kiểu nguyên [x1, y1, x2, y2] trong khung hình letterbox.

        Returns:
          - crop_boxes: np.ndarray [T, 4] dtype=np.int32: [x1, y1, x2, y2]
          - smoothed_boxes: np.ndarray [T, 4] dtype=np.float32: tọa độ float sau EMA
        """
        T = len(chosen_bboxes)
        target_w, target_h = self.letterbox_size
        raw_boxes = np.zeros((T, 4), dtype=np.float32)

        for t in range(T):
            if found[t] and chosen_bboxes[t] is not None:
                bx1, by1, bx2, by2 = chosen_bboxes[t][:4]
                lms = landmarks_px[t]  # [68, 2]
                lx1, ly1 = float(np.min(lms[:, 0])), float(np.min(lms[:, 1]))
                lx2, ly2 = float(np.max(lms[:, 0])), float(np.max(lms[:, 1]))

                # Hợp nhất hộp dò mặt và cực trị điểm mốc để không lẹm cằm/mắt
                fx1 = min(bx1, lx1)
                fy1 = min(by1, ly1)
                fx2 = max(bx2, lx2)
                fy2 = max(by2, ly2)

                fw = fx2 - fx1
                fh = fy2 - fy1
                cx1 = fx1 - fw * self.bbox_margin
                cy1 = fy1 - fh * self.bbox_margin
                cx2 = fx2 + fw * self.bbox_margin
                cy2 = fy2 + fh * self.bbox_margin

                # Giữ khung vuông quanh tâm khuôn mặt
                center_x = (cx1 + cx2) / 2.0
                center_y = (cy1 + cy2) / 2.0
                side = max(cx2 - cx1, cy2 - cy1)

                raw_boxes[t] = [
                    center_x - side / 2.0,
                    center_y - side / 2.0,
                    center_x + side / 2.0,
                    center_y + side / 2.0
                ]

        # 2. Nội suy tuyến tính cho các frame mất mặt
        valid_indices = [t for t in range(T) if found[t]]
        if not valid_indices:
            side = min(target_w, target_h) * 0.8
            cx, cy = target_w / 2.0, target_h / 2.0
            default_box = np.array([cx - side / 2.0, cy - side / 2.0, cx + side / 2.0, cy + side / 2.0],
                                   dtype=np.float32)
            for t in range(T):
                raw_boxes[t] = default_box
        else:
            first_idx = valid_indices[0]
            for t in range(first_idx):
                raw_boxes[t] = raw_boxes[first_idx]
            last_idx = valid_indices[-1]
            for t in range(last_idx + 1, T):
                raw_boxes[t] = raw_boxes[last_idx]
            for i in range(len(valid_indices) - 1):
                idx_a = valid_indices[i]
                idx_b = valid_indices[i + 1]
                if idx_b - idx_a > 1:
                    box_a = raw_boxes[idx_a]
                    box_b = raw_boxes[idx_b]
                    for t in range(idx_a + 1, idx_b):
                        alpha = float(t - idx_a) / float(idx_b - idx_a)
                        raw_boxes[t] = (1.0 - alpha) * box_a + alpha * box_b

        # 3. Làm mượt thời gian bằng bộ lọc EMA
        smoothed_boxes = np.zeros_like(raw_boxes)
        smoothed_boxes[0] = raw_boxes[0]
        alpha = self.box_smooth_alpha
        for t in range(1, T):
            smoothed_boxes[t] = alpha * raw_boxes[t] + (1.0 - alpha) * smoothed_boxes[t - 1]

        # 4. Giới hạn tọa độ trong kích thước khung letterbox và ép kiểu int32
        crop_boxes = np.zeros((T, 4), dtype=np.int32)
        for t in range(T):
            bx1, by1, bx2, by2 = smoothed_boxes[t]
            x1 = max(0, min(target_w - 2, int(round(bx1))))
            y1 = max(0, min(target_h - 2, int(round(by1))))
            x2 = max(x1 + 1, min(target_w, int(round(bx2))))
            y2 = max(y1 + 1, min(target_h, int(round(by2))))
            crop_boxes[t] = [x1, y1, x2, y2]

        return crop_boxes, smoothed_boxes

    def _crop_faces_for_litefat(
            self,
            frames: List[np.ndarray],
            crop_boxes: np.ndarray,
    ) -> torch.Tensor:
        """
        Cắt vùng khuôn mặt chuẩn hóa 224x224 cho nhánh MobileNetV3-Small từ crop_boxes.
        """
        return crop_faces_from_boxes(frames, crop_boxes, face_size=self.face_size)

    def extract_from_frames(
            self,
            frames: List[np.ndarray],
            crop_faces: bool = True,
            return_crop_boxes: bool = False,
    ) -> Union[
        Tuple[Optional[torch.Tensor], torch.Tensor, List[Optional[np.ndarray]], List[bool]],
        Tuple[Optional[torch.Tensor], torch.Tensor, List[Optional[np.ndarray]], List[bool], np.ndarray],
    ]:
        """
        Pipeline trích xuất hoàn chỉnh từ chuỗi khung hình đã qua letterbox:
          - Phát hiện mặt batched GPU.
          - Chọn khuôn mặt tài xế ổn định.
          - Trích xuất 68 điểm mốc batched GPU.
          - Tính tọa độ điểm cắt khung mặt và (tùy chọn) cắt/chuẩn hóa khung hình mặt.

        Returns:
          - face_frames: Tensor [T, 3, 224, 224] (hoặc None nếu crop_faces=False)
          - landmarks:   Tensor [T, 68, 2] (X, Y)
          - bboxes:      Danh sách BBox chọn lọc của từng frame
          - found:       Danh sách boolean đánh dấu phát hiện được mặt
          - crop_boxes:  (kèm theo nếu return_crop_boxes=True) mảng np.ndarray [T, 4] dạng [x1, y1, x2, y2]
        """
        target_w, target_h = self.letterbox_size
        # Đảm bảo mọi frame đầu vào đúng kích thước letterbox
        uniform_frames = []
        for f in frames:
            if f.shape[1] != target_w or f.shape[0] != target_h:
                uniform_frames.append(letterbox(f, new_size=(target_w, target_h)))
            else:
                uniform_frames.append(f)

        # 1. Phát hiện mặt theo lô
        all_dets = self._detect_faces_batched(uniform_frames)

        # 2. Lựa chọn khuôn mặt tài xế ổn định
        chosen_bboxes, found = self._select_driver_faces(all_dets, target_w, target_h)

        # 3. Trích xuất 68 điểm mốc theo lô
        landmarks_norm, landmarks_px = self._extract_landmarks_batched(
            uniform_frames, chosen_bboxes, found
        )

        # 4. Tính toán tọa độ điểm cắt khuôn mặt
        crop_boxes, _ = self.compute_face_crop_boxes(chosen_bboxes, landmarks_px, found)

        # 5. Cắt khung hình khuôn mặt nếu cần
        face_frames_tensor = None
        if crop_faces:
            face_frames_tensor = self._crop_faces_for_litefat(uniform_frames, crop_boxes)

        landmarks_tensor = torch.from_numpy(landmarks_norm).float()

        if return_crop_boxes:
            return face_frames_tensor, landmarks_tensor, chosen_bboxes, found, crop_boxes
        return face_frames_tensor, landmarks_tensor, chosen_bboxes, found


# =====================================================================
# 2. ĐỌC NHANH KHUNG HÌNH TỪ VIDEO (FAST VIDEO SEEKER)
# =====================================================================
def fast_read_video_frames(
        video_path: Union[str, Path],
        target_indices: np.ndarray,
        letterbox_size: Tuple[int, int] = (640, 480),
) -> Tuple[List[np.ndarray], float, int]:
    """
    Đọc chính xác các chỉ số khung hình `target_indices` từ video với tốc độ cao:
      - Nhảy thẳng bằng CAP_PROP_POS_FRAMES tới vị trí đầu tiên.
      - Dùng cap.grab() để lướt qua các frame trung gian (tiết kiệm chi phí giải mã CPU).
      - Áp dụng ultis.letterbox về đúng kích thước chuẩn (target_w, target_h).
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise IOError(f"Không thể mở tệp video: {video_path}")

    fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    target_w, target_h = letterbox_size

    frames: List[np.ndarray] = []
    curr_frame = int(target_indices[0])
    cap.set(cv2.CAP_PROP_POS_FRAMES, curr_frame)

    try:
        for target_idx in target_indices:
            target_idx = int(target_idx)
            if target_idx < curr_frame and frames:
                frames.append(frames[-1].copy())
                continue

            while curr_frame < target_idx:
                if not cap.grab():
                    break
                curr_frame += 1

            ret, frame = cap.read()
            if not ret or frame is None:
                break

            # Chuẩn hóa qua ultis.letterbox
            lb_frame = letterbox(frame, new_size=(target_w, target_h))
            frames.append(lb_frame)
            curr_frame += 1
    finally:
        cap.release()

    if not frames:
        raise IOError(f"Không đọc được khung hình nào từ: {video_path}")

    # Bù khung hình nếu video kết thúc sớm hơn số lượng yêu cầu
    while len(frames) < len(target_indices):
        frames.append(frames[-1].copy())

    return frames, fps, total_frames


# =====================================================================
# 3. LỚP LITEFAT DATASET (HỖ TRỢ ĐA CHẾ ĐỘ LẤY MẪU & CACHE)
# =====================================================================
class LiteFatDataset(Dataset):
    """
    PyTorch Dataset nạp dữ liệu cho mô hình LiteFat từ E:\\LSTM\\data_processed:
      - Hỗ trợ nạp từ manifest dataset_merged_split.csv hoặc quét thư mục train/val/test.
      - Hỗ trợ 2 chế độ lấy mẫu thời gian:
          + Chế độ 1 (mặc định): Cố định num_frames (T=16) theo bài báo LiteFat.
            Train: Temporal Jitter (lấy ngẫu nhiên trong từng phân đoạn); Val/Test: Linspace đều đặn.
          + Chế độ 2: Lấy mẫu theo time_interval và window_sec.
      - Tiền xử lý toàn bộ khung hình qua ultis.letterbox (640, 480).
      - Trích xuất 68 điểm mốc bằng FastPIPNetExtractor (Batched GPU).
      - Cắt khung hình khuôn mặt 224x224 chuẩn hóa ImageNet cho MobileNetV3-Small.
      - Hỗ trợ nạp / lưu cache .pt để tăng tốc tối đa khi huấn luyện nhiều epoch.
    """

    def __init__(
            self,
            dataset_dir: Union[str, Path] = r"E:\LSTM\data_processed",
            split: str = "train",
            time_interval: float = 0.1,
            window_sec: Optional[float] = 10.0,
            manifest_filename: str = "dataset_merged_split.csv",
            letterbox_size: Tuple[int, int] = (640, 640),
            face_size: Tuple[int, int] = (224, 224),
            conf_threshold: float = 0.5,
            bbox_margin: float = 0.15,
            return_meta: bool = False,
            return_raw_frames: bool = False,
            use_cache: bool = False,
            cache_dir: Optional[Union[str, Path]] = None,
            extractor: Optional[FastPIPNetExtractor] = None,
            device: Optional[str] = None,
            extract_crops_only: bool = False,
            pipnet_backbone: str = "resnet18",
            nms_threshold: float = 0.3,
            box_smooth_alpha: float = 0.7,
            selected_datasets: Optional[Union[List[str], str]] = None,
            use_visual_context: bool = True,
            augment_train: bool = True,
            flip_prob: float = 0.5,
            scale_jitter: float = 0.08,
            shift_jitter: float = 0.05,
            roll_deg: float = 12.0,
            temporal_dropout: float = 0.05,
            landmark_noise_std: float = 0.002,
            color_jitter: float = 0.2,
            spatial_dropout: float = 0.15,
    ):
        split = split.lower().strip()
        if split not in ("train", "val", "test"):
            raise ValueError(f"Split không hợp lệ: '{split}'. Chỉ chấp nhận 'train', 'val', hoặc 'test'.")

        if time_interval is not None and float(time_interval) < 0:
            raise ValueError(f"time_interval phải là số không âm (>= 0): {time_interval}")
        if window_sec is not None and float(window_sec) <= 0:
            raise ValueError(f"window_sec phải là số dương hoặc None: {window_sec}")

        self.root = Path(dataset_dir)
        self.split = split
        self.time_interval = float(time_interval) if time_interval is not None else 0.1
        self.window_sec = float(window_sec) if window_sec is not None else None
        self.manifest_filename = manifest_filename
        self.letterbox_size = _parse_size(letterbox_size)
        self.face_size = _parse_size(face_size)
        self.conf_threshold = conf_threshold
        self.nms_threshold = nms_threshold
        self.bbox_margin = bbox_margin
        self.box_smooth_alpha = box_smooth_alpha
        self.pipnet_backbone = pipnet_backbone
        self.device = device
        self.return_meta = bool(return_meta)
        self.return_raw_frames = bool(return_raw_frames)
        self.use_cache = bool(use_cache)
        self.cache_dir = Path(cache_dir) if cache_dir is not None else None

        # Cấu hình bối cảnh thị giác (MobileNetV3) & Đa phương thức
        self.use_visual_context = bool(use_visual_context)
        self.extract_crops_only = bool(extract_crops_only) or (not self.use_visual_context)

        # Cấu hình lọc bộ dữ liệu con
        if selected_datasets is not None:
            if isinstance(selected_datasets, str):
                self.selected_datasets = [selected_datasets.lower()]
            else:
                self.selected_datasets = [str(s).lower() for s in selected_datasets if str(s).strip()]
                if not self.selected_datasets or any(s in ("all", "*") for s in self.selected_datasets):
                    self.selected_datasets = None
        else:
            self.selected_datasets = None

        # Cấu hình tăng cường dữ liệu
        self.augment_train = bool(augment_train)
        self.flip_prob = float(flip_prob)
        self.scale_jitter = float(scale_jitter)
        self.shift_jitter = float(shift_jitter)
        self.roll_deg = float(roll_deg)
        self.temporal_dropout = float(temporal_dropout)
        self.landmark_noise_std = float(landmark_noise_std)
        self.color_jitter = float(color_jitter)
        self.spatial_dropout = float(spatial_dropout)

        self.cache_meta: Dict[str, Any] = {}
        self.cached_data: Optional[Dict[str, Any]] = None

        # Nạp cache từ file .pt nếu được yêu cầu
        if self.use_cache and self.cache_dir is not None:
            self._load_cache()

        # Nạp danh sách mẫu (ưu tiên manifest CSV hoặc quét thư mục, nếu rỗng thì lấy từ cache)
        self.samples = self._load_samples()
        if len(self.samples) == 0:
            if self.cached_data is not None and len(self.cached_data) > 0:
                logger.info(
                    f"[Cache] Tự động khởi tạo mẫu từ file cache .pt (selected_datasets={self.selected_datasets}).")
                for k, it in self.cached_data.items():
                    m = it.get("meta", {})
                    rel_p = m.get("key", k)
                    if not self._is_dataset_selected(rel_p, m.get("dataset")):
                        continue
                    self.samples.append({
                        "key": rel_p,
                        "path": self.root / rel_p,
                        "label": int(it.get("label", m.get("label", 0))),
                        "video_id": m.get("video_id", Path(rel_p).stem),
                        "fps": float(m.get("fps", 30.0)),
                        "frame_count": int(float(m.get("total_frames", 0))),
                        "duration_sec": float(m.get("duration_sec", 0.0)),
                    })
                logger.info(f"[Cache] Số mẫu sau lọc dataset: {len(self.samples)}")
            else:
                logger.warning(f"Không tìm thấy mẫu video nào trong {self.root} cho split='{self.split}'")

        # Khởi tạo bộ trích xuất PIPNet:
        # Nếu đã có cache và không cung cấp extractor từ ngoài, trì hoãn (Lazy Init)
        # để không tốn VRAM tải FaceBoxesV2 + PIPNet khi chỉ nạp từ cache.
        self.extractor = extractor
        if self.extractor is None and not (self.use_cache and self.cached_data is not None):
            self.extractor = self._get_or_create_extractor()

    def _get_or_create_extractor(self) -> FastPIPNetExtractor:
        """Khởi tạo trễ (Lazy init) FastPIPNetExtractor khi thực sự cần trích xuất trực tiếp."""
        if self.extractor is None:
            self.extractor = FastPIPNetExtractor(
                backbone=self.pipnet_backbone,
                device=self.device,
                letterbox_size=self.letterbox_size,
                face_size=self.face_size,
                conf_threshold=self.conf_threshold,
                nms_threshold=self.nms_threshold,
                bbox_margin=self.bbox_margin,
                box_smooth_alpha=self.box_smooth_alpha,
            )
        return self.extractor

    def _is_dataset_selected(self, key: str, dataset_field: Optional[str] = None) -> bool:
        """Kiểm tra xem mẫu có thuộc danh sách selected_datasets được chọn hay không."""
        if self.selected_datasets is None:
            return True
        key_lower = key.lower()
        if dataset_field and any(ds in dataset_field.lower() for ds in self.selected_datasets):
            return True
        return any(ds in key_lower for ds in self.selected_datasets)

    def _load_samples(self) -> List[Dict[str, Any]]:
        """Đọc danh sách mẫu video từ manifest CSV hoặc quét thư mục fallback."""
        manifest_path = self.root / self.manifest_filename
        samples = []

        if manifest_path.is_file():
            with open(manifest_path, mode="r", encoding="utf-8-sig") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    if row.get("split", "").strip().lower() == self.split:
                        rel_path = row.get("rel_path") or row.get("processed_path")
                        if not rel_path:
                            continue
                        norm_rel = Path(rel_path.replace("\\", "/")).as_posix()
                        if not self._is_dataset_selected(norm_rel, row.get("dataset")):
                            continue
                        full_path = self.root / norm_rel
                        samples.append({
                            "key": norm_rel,
                            "path": full_path,
                            "label": int(row["label"]),
                            "video_id": row.get("video_id") or Path(rel_path).stem,
                            "subject_id": str(row.get("subject_id", "") or "unknown"),
                            "dataset": str(row.get("dataset", "") or "unknown"),
                            "fps": float(row.get("fps", 30.0)),
                            "frame_count": int(float(row.get("frame_count", 0))),
                            "duration_sec": float(row.get("duration_sec", 0.0)),
                        })
        else:
            # Fallback quét thư mục train/val/test/0_alert và 1_drowsy
            for label, folder in ((0, "0_alert"), (1, "1_drowsy")):
                split_folder = self.root / self.split / folder
                if split_folder.is_dir():
                    for p in sorted(split_folder.glob("*.mp4")):
                        rel_key = p.relative_to(self.root).as_posix()
                        if not self._is_dataset_selected(rel_key):
                            continue
                        samples.append({
                            "key": rel_key,
                            "path": p,
                            "label": label,
                            "video_id": p.stem,
                            "subject_id": "unknown",
                            "dataset": "unknown",
                            "fps": 30.0,
                            "frame_count": 0,
                            "duration_sec": 0.0,
                        })
        return samples

    def _load_cache(self) -> None:
        """Nạp dữ liệu cache từ tệp .pt và lưu trữ metadata của cache."""
        cache_path = self.cache_dir / f"{self.split}.pt"
        if not cache_path.is_file():
            logger.warning(f"[Cache] Không tìm thấy tệp cache {cache_path}. Sẽ fallback trích xuất on-the-fly.")
            return

        try:
            payload = torch.load(cache_path, map_location="cpu", weights_only=False)
            if isinstance(payload, dict) and "samples" in payload:
                self.cached_data = payload["samples"]
                self.cache_meta = {
                    "format_version": payload.get("format_version", 1),
                    "split": payload.get("split", self.split),
                    "time_interval": float(payload.get("time_interval", 0.05)),
                    "window_sec": payload.get("window_sec", None),
                    "letterbox_size": tuple(payload.get("letterbox_size", self.letterbox_size)),
                    "face_size": tuple(payload.get("face_size", self.face_size)),
                    "bbox_margin": float(payload.get("bbox_margin", self.bbox_margin)),
                    "box_smooth_alpha": float(payload.get("box_smooth_alpha", self.box_smooth_alpha)),
                    "num_samples": payload.get("num_samples", len(self.cached_data)),
                }
            elif isinstance(payload, dict):
                self.cached_data = payload
                self.cache_meta = {
                    "time_interval": 0.05,
                    "window_sec": None,
                    "letterbox_size": self.letterbox_size,
                    "face_size": self.face_size,
                }

            cached_lbs = self.cache_meta.get("letterbox_size")
            if cached_lbs and tuple(cached_lbs) != tuple(self.letterbox_size):
                logger.warning(
                    f"[Cache] File {cache_path.name} được trích xuất với letterbox_size={cached_lbs}, "
                    f"khác với cấu hình hiện tại {self.letterbox_size}. "
                    f"Sẽ sử dụng letterbox_size của cache để bảo đảm tọa độ crop_boxes khớp tuyệt đối."
                )

            logger.info(f"[Cache] Đã nạp thành công cache {cache_path.name} với {len(self.cached_data)} mẫu.")
        except Exception as e:
            logger.warning(f"[Cache] Lỗi khi nạp cache {cache_path}: {e}. Bỏ qua cache.")

    def _get_frame_indices(self, total_frames: int, fps: float) -> np.ndarray:
        """
        Tính toán danh sách chỉ số khung hình cần lấy từ video:
          - window_sec = None -> Cửa sổ trượt là TOÀN BỘ video (0 đến total_frames - 1).
          - window_sec > 0    -> Cửa sổ trượt dài window_sec giây (Random crop khi train, Center crop khi val/test).
          - time_interval == 0 -> Lấy liên tiếp mọi khung hình trong cửa sổ (không có khoảng cách thời gian giữa các frame).
          - time_interval > 0  -> Lấy mẫu cách đều sau mỗi time_interval giây trên trục thời gian thực.
        """
        if total_frames <= 0:
            return np.array([0], dtype=int)

        fps = fps if fps > 0 else 30.0
        duration_sec = total_frames / fps

        # TRƯỜNG HỢP 1: time_interval == 0 (Lấy các khung hình liên tiếp không bỏ sót frame nào)
        if self.time_interval == 0:
            if self.window_sec is None:
                # window_sec = None: lấy toàn bộ video liên tiếp
                return np.arange(total_frames, dtype=int)
            else:
                win_sec = min(self.window_sec, duration_sec)
                num_consec = int(round(win_sec * fps))
                num_consec = max(1, min(num_consec, total_frames))
                max_start = max(0, total_frames - num_consec)
                start_idx = random.randint(0, max_start) if (
                        self.split == "train" and max_start > 0) else max_start // 2
                return np.arange(start_idx, start_idx + num_consec, dtype=int)

        # TRƯỜNG HỢP 2: time_interval > 0 (Lấy mẫu cách đều theo thời gian thực)
        if self.window_sec is None:
            # window_sec = None: toàn bộ video cách đều mỗi time_interval giây
            timestamps = np.arange(0, duration_sec, self.time_interval)
            if len(timestamps) == 0:
                timestamps = np.array([0.0])
            indices = np.rint(timestamps * fps).astype(int)
            return np.clip(indices, 0, total_frames - 1)
        else:
            win = min(self.window_sec, duration_sec)
            max_start = max(0.0, duration_sec - win)
            start_sec = random.uniform(0.0, max_start) if (self.split == "train" and max_start > 0) else max_start / 2.0
            timestamps = start_sec + np.arange(0, win, self.time_interval)
            if len(timestamps) == 0:
                timestamps = np.array([start_sec])
            indices = np.rint(timestamps * fps).astype(int)
            return np.clip(indices, 0, total_frames - 1)

    def _slice_cached_sequence(self, item: Dict[str, Any], fps: float) -> np.ndarray:
        """
        Xác định các chỉ số khung hình (pos) cần lấy từ chuỗi cache .pt:
          - Dựa vào time_interval và window_sec được cấu hình trong LiteFatDataset.
          - Tự động phát hiện khoảng cách thời gian giữa các frame trong cache (cache_dt).
          - Hỗ trợ cắt ngẫu nhiên (train) và cắt chính giữa (val/test).
        """
        landmarks = item["landmarks"]
        T_cache = int(landmarks.shape[0])
        if T_cache <= 0:
            return np.array([0], dtype=int)

        cache_dt = float(self.cache_meta.get("time_interval", 0.05))
        if cache_dt <= 0:
            cache_dt = 1.0 / (fps if fps > 0 else 30.0)

        meta = item.get("meta", {})
        duration_sec = float(meta.get("duration_sec", T_cache * cache_dt))
        if duration_sec <= 0:
            duration_sec = T_cache * cache_dt

        # TRƯỜNG HỢP 1: time_interval == 0 (Lấy chuỗi khung hình liên tiếp)
        if self.time_interval == 0:
            if self.window_sec is None:
                return np.arange(T_cache, dtype=int)
            else:
                win_sec = min(self.window_sec, duration_sec)
                num_consec = int(round(win_sec / cache_dt))
                num_consec = max(1, min(num_consec, T_cache))
                max_start = max(0, T_cache - num_consec)
                start_idx = random.randint(0, max_start) if (
                        self.split == "train" and max_start > 0) else max_start // 2
                return np.arange(start_idx, start_idx + num_consec, dtype=int)

        # TRƯỜNG HỢP 2: time_interval > 0 (Lấy mẫu cách đều theo thời gian)
        if self.window_sec is None:
            # Nếu time_interval cấu hình trùng khớp với cache_dt: giữ nguyên toàn bộ cache
            if abs(self.time_interval - cache_dt) < 1e-5:
                return np.arange(T_cache, dtype=int)
            timestamps = np.arange(0, duration_sec, self.time_interval)
            if len(timestamps) == 0:
                timestamps = np.array([0.0])
            pos = np.rint(timestamps / cache_dt).astype(int)
            return np.clip(pos, 0, T_cache - 1)
        else:
            if self.split != "train":
                # Đối với tập val và test: Đánh giá trên tối đa 18.0 giây (360 frames)
                # để bao quát trọn vẹn toàn bộ chuỗi video UL-DD/VBDDD, không bị cắt cụt mất sự kiện buồn ngủ
                eval_win = min(max(self.window_sec or 10.0, 18.0), duration_sec)
                timestamps = np.arange(0, eval_win, self.time_interval)
                if len(timestamps) == 0:
                    timestamps = np.array([0.0])
                pos = np.rint(timestamps / cache_dt).astype(int)
                return np.clip(pos, 0, T_cache - 1)
            else:
                win = min(self.window_sec, duration_sec)
                max_start = max(0.0, duration_sec - win)
                start_sec = random.uniform(0.0, max_start) if max_start > 0 else 0.0
                timestamps = start_sec + np.arange(0, win, self.time_interval)
                if len(timestamps) == 0:
                    timestamps = np.array([start_sec])
                pos = np.rint(timestamps / cache_dt).astype(int)
                return np.clip(pos, 0, T_cache - 1)

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> Union[
        Tuple[Optional[torch.Tensor], torch.Tensor, torch.Tensor],
        Tuple[Optional[torch.Tensor], torch.Tensor, torch.Tensor, Dict[str, Any]],
        Tuple[Optional[torch.Tensor], torch.Tensor, torch.Tensor, Dict[str, Any], List[np.ndarray]],
    ]:
        sample = self.samples[index]
        key = sample["key"]

        # 1. Đường nhanh: Nạp từ RAM Cache nếu có
        if self.cached_data is not None and key in self.cached_data:
            item = self.cached_data[key]
            fps = float(sample.get("fps") or item.get("meta", {}).get("fps", 30.0))
            pos = self._slice_cached_sequence(item, fps)

            # Cắt các đặc trưng mốc và hộp giới hạn theo pos (tương thích cache 2 kênh hoặc 3 kênh cũ)
            landmarks = item["landmarks"][pos]
            if landmarks.shape[-1] > 2:
                landmarks = landmarks[..., :2]
            crop_boxes = item["crop_boxes"][pos] if "crop_boxes" in item else None
            bboxes = item["bboxes"][pos] if "bboxes" in item else None
            found = item["found"][pos] if "found" in item else None

            # Lấy target_indices tương ứng trong video gốc
            cached_target_indices = item.get("target_indices")
            if cached_target_indices is None and "meta" in item and isinstance(item["meta"], dict):
                cached_target_indices = item["meta"].get("target_indices")

            if cached_target_indices is not None:
                if isinstance(cached_target_indices, torch.Tensor):
                    selected_target_indices = cached_target_indices[pos].long().cpu().numpy()
                else:
                    selected_target_indices = np.asarray(cached_target_indices, dtype=int)[pos]
            else:
                cache_dt = float(self.cache_meta.get("time_interval", 0.05))
                selected_target_indices = np.rint(pos * (cache_dt * fps)).astype(int)

            cache_lbs = self.cache_meta.get("letterbox_size", self.letterbox_size)
            face_frames = item.get("face_frames", None)
            if face_frames is not None:
                face_frames = face_frames[pos]

            raw_frames: Optional[List[np.ndarray]] = None

            # Đọc video nếu cần tạo face_frames (cho MobileNetV3) hoặc người dùng yêu cầu return_raw_frames
            need_video_read = (not self.extract_crops_only and face_frames is None) or self.return_raw_frames
            if need_video_read:
                raw_frames, actual_fps, _ = fast_read_video_frames(
                    sample["path"], selected_target_indices, letterbox_size=cache_lbs
                )
                if not self.extract_crops_only and face_frames is None and crop_boxes is not None:
                    face_frames = crop_faces_from_boxes(raw_frames, crop_boxes, face_size=self.face_size)

            if not self.use_visual_context:
                face_frames = None

            if self.split == "train" and self.augment_train:
                face_frames, landmarks = augment_multimodal_clip(
                    face_frames,
                    landmarks,
                    is_train=True,
                    flip_prob=self.flip_prob,
                    scale_jitter=self.scale_jitter,
                    shift_jitter=self.shift_jitter,
                    roll_deg=self.roll_deg,
                    temporal_dropout=self.temporal_dropout,
                    landmark_noise_std=self.landmark_noise_std,
                    color_jitter=self.color_jitter,
                    spatial_dropout=self.spatial_dropout,
                )

            label = torch.tensor(sample["label"], dtype=torch.long)

            if found is not None:
                found_t = torch.as_tensor(found, dtype=torch.bool)
                found_ratio = float(found_t.sum().item()) / float(max(1, len(found_t)))
            else:
                found_t = torch.ones(len(pos), dtype=torch.bool)
                found_ratio = 1.0

            meta = {
                "key": key,
                "video_id": sample["video_id"],
                "subject_id": str(sample.get("subject_id") or (item.get("meta", {}).get("subject_id", "unknown"))),
                "dataset": str(sample.get("dataset") or (item.get("meta", {}).get("dataset", "unknown"))),
                "path": str(sample["path"]),
                "split": self.split,
                "label": int(sample["label"]),
                "fps": float(fps),
                "total_frames": int(sample.get("frame_count") or item.get("meta", {}).get("total_frames", 0)),
                "duration_sec": float(sample.get("duration_sec") or item.get("meta", {}).get("duration_sec", 0.0)),
                "target_indices": torch.as_tensor(selected_target_indices, dtype=torch.long),
                "num_frames": int(landmarks.shape[0]),
                "found_faces_ratio": found_ratio,
                "letterbox_size": cache_lbs,
                "cached": True,
            }
            if crop_boxes is not None:
                meta["crop_boxes"] = torch.as_tensor(crop_boxes, dtype=torch.int32)
            if bboxes is not None:
                meta["bboxes"] = torch.as_tensor(bboxes, dtype=torch.float32)
            if found is not None:
                meta["found"] = found_t

            if "meta" in item and isinstance(item["meta"], dict):
                for k_meta, v_meta in item["meta"].items():
                    if k_meta not in meta and k_meta != "path":
                        meta[k_meta] = v_meta
                if "path" in item["meta"] and str(item["meta"]["path"]) != str(sample["path"]):
                    meta["remote_path"] = str(item["meta"]["path"])

            if self.return_raw_frames and self.return_meta:
                return face_frames, landmarks, label, meta, (raw_frames if raw_frames is not None else [])
            if self.return_meta:
                return face_frames, landmarks, label, meta
            return face_frames, landmarks, label

        # 2. Đường chuẩn: Đọc video và trích xuất on-the-fly
        extractor = self._get_or_create_extractor()
        fps = sample["fps"]
        total_frames = sample["frame_count"]

        if fps <= 0 or total_frames <= 0:
            cap = cv2.VideoCapture(str(sample["path"]))
            fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 300)
            cap.release()
            sample["fps"] = fps
            sample["frame_count"] = total_frames

        target_indices = self._get_frame_indices(total_frames, fps)
        raw_frames, actual_fps, _ = fast_read_video_frames(
            sample["path"], target_indices, letterbox_size=self.letterbox_size
        )

        crop_faces = not self.extract_crops_only
        face_frames, landmarks, bboxes, found, crop_boxes = extractor.extract_from_frames(
            raw_frames,
            crop_faces=crop_faces,
            return_crop_boxes=True,
        )

        if not self.use_visual_context:
            face_frames = None

        if self.split == "train" and self.augment_train:
            face_frames, landmarks = augment_multimodal_clip(
                face_frames,
                landmarks,
                is_train=True,
                flip_prob=self.flip_prob,
                scale_jitter=self.scale_jitter,
                shift_jitter=self.shift_jitter,
                roll_deg=self.roll_deg,
                temporal_dropout=self.temporal_dropout,
                landmark_noise_std=self.landmark_noise_std,
                color_jitter=self.color_jitter,
                spatial_dropout=self.spatial_dropout,
            )

        label = torch.tensor(sample["label"], dtype=torch.long)

        # Chuẩn hóa BBoxes và Found thành Tensor để dễ lưu .pt và xử lý
        T_frames = len(found)
        bboxes_tensor = torch.zeros((T_frames, 5), dtype=torch.float32)
        for t_idx, b in enumerate(bboxes):
            if b is not None and len(b) >= 4:
                b_arr = np.asarray(b, dtype=np.float32)
                bboxes_tensor[t_idx, :min(5, len(b_arr))] = torch.from_numpy(b_arr[:min(5, len(b_arr))])
        found_tensor = torch.tensor(found, dtype=torch.bool)
        crop_boxes_tensor = torch.from_numpy(crop_boxes).int()

        meta = {
            "key": key,
            "video_id": sample["video_id"],
            "subject_id": str(sample.get("subject_id", "unknown")),
            "dataset": str(sample.get("dataset", "unknown")),
            "path": str(sample["path"]),
            "split": self.split,
            "label": int(sample["label"]),
            "fps": float(actual_fps),
            "total_frames": int(total_frames),
            "duration_sec": float(total_frames / actual_fps) if actual_fps > 0 else 0.0,
            "target_indices": torch.from_numpy(target_indices).long(),
            "num_frames": int(landmarks.shape[0]),
            "found_faces_ratio": float(sum(found)) / float(max(1, len(found))),
            "letterbox_size": self.letterbox_size,
            "crop_boxes": crop_boxes_tensor,
            "bboxes": bboxes_tensor,
            "found": found_tensor,
            "raw_bboxes_list": bboxes,
            "cached": False,
        }

        if self.return_raw_frames and self.return_meta:
            return face_frames, landmarks, label, meta, raw_frames
        if self.return_meta:
            return face_frames, landmarks, label, meta
        return face_frames, landmarks, label


# =====================================================================
# 4. HÀM COLLATE & XÂY DỰNG DATALOADER
# =====================================================================
def collate_litefat(batch: List[Tuple[Any, ...]]):
    """
    Collate function cho DataLoader LiteFat.
    Hỗ trợ tự động đệm (padding) theo chiều thời gian T nếu độ dài giữa các mẫu khác nhau
    (khi sử dụng chế độ num_frames=None, time_interval).
    Hỗ trợ cả trường hợp face_frames là None (khi chạy ở chế độ extract_crops_only=True).
    """
    first_item = batch[0]
    num_elements = len(first_item)

    face_frames_list = [item[0] for item in batch]
    landmarks_list = [item[1] for item in batch]
    labels_list = [item[2] for item in batch]

    lengths = [lm.shape[0] for lm in landmarks_list]
    max_len = max(lengths)

    has_face_frames = all(f is not None for f in face_frames_list)

    # Đệm theo chiều thời gian T nếu các video có số frame khác nhau
    padded_frames = []
    padded_landmarks = []
    for idx, (lm, cur_len) in enumerate(zip(landmarks_list, lengths)):
        f = face_frames_list[idx]
        if cur_len < max_len:
            pad_t = max_len - cur_len
            if has_face_frames and f is not None:
                # Đệm frame bằng frame cuối cùng
                last_frame = f[-1:].repeat(pad_t, 1, 1, 1)
                padded_f = torch.cat([f, last_frame], dim=0)
                padded_frames.append(padded_f)
            # Đệm landmark bằng giá trị 1.0 (chuẩn bài báo LiteFat Section III-B)
            pad_lm = torch.ones((pad_t, lm.shape[1], lm.shape[2]), dtype=lm.dtype)
            padded_l = torch.cat([lm, pad_lm], dim=0)
            padded_landmarks.append(padded_l)
        else:
            if has_face_frames and f is not None:
                padded_frames.append(f)
            padded_landmarks.append(lm)

    batch_frames = torch.stack(padded_frames, dim=0) if has_face_frames else None  # [B, T, 3, 224, 224] hoặc None
    batch_landmarks = torch.stack(padded_landmarks, dim=0)  # [B, T, 68, 2]
    batch_labels = torch.stack(labels_list, dim=0)  # [B]
    batch_lengths = torch.tensor(lengths, dtype=torch.long)  # [B]

    if num_elements >= 4:
        metas = [item[3] for item in batch]
        if num_elements == 5:
            raw_frames_list = [item[4] for item in batch]
            return batch_frames, batch_landmarks, batch_labels, batch_lengths, metas, raw_frames_list
        return batch_frames, batch_landmarks, batch_labels, batch_lengths, metas

    return batch_frames, batch_landmarks, batch_labels, batch_lengths


def build_litefat_dataloaders(
        dataset_dir: Union[str, Path] = r"E:\LSTM\data_processed",
        batch_size: int = 8,
        num_workers: int = 0,
        time_interval: float = 0.05,
        window_sec: Optional[float] = 15.0,
        letterbox_size: Tuple[int, int] = (640, 640),
        face_size: Tuple[int, int] = (224, 224),
        manifest_filename: str = "dataset_merged_split.csv",
        use_cache: bool = False,
        cache_dir: Optional[Union[str, Path]] = None,
        pin_memory: bool = True,
        shuffle_train: bool = True,
        device: Optional[str] = None,
        pipnet_backbone: str = "resnet18",
        conf_threshold: float = 0.5,
        nms_threshold: float = 0.3,
        bbox_margin: float = 0.15,
        box_smooth_alpha: float = 0.7,
        extract_crops_only: bool = False,
        selected_datasets: Optional[Union[List[str], str]] = None,
        use_visual_context: bool = True,
        augment_train: bool = True,
        flip_prob: float = 0.5,
        scale_jitter: float = 0.08,
        shift_jitter: float = 0.05,
        roll_deg: float = 12.0,
        temporal_dropout: float = 0.05,
        landmark_noise_std: float = 0.0003,
        color_jitter: float = 0.2,
        spatial_dropout: float = 0.15,
        subject_balanced_sampling: bool = True,
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """
    Khởi tạo nhanh bộ 3 DataLoader cho tập train, val, test của mô hình LiteFat theo cơ chế window_sec và time_interval.
    """
    # Khởi tạo 1 extractor dùng chung cho cả 3 tập nếu không dùng cache để tiết kiệm VRAM và tăng tốc khởi tạo
    shared_extractor = FastPIPNetExtractor(
        backbone=pipnet_backbone,
        device=device,
        letterbox_size=letterbox_size,
        face_size=face_size,
        conf_threshold=conf_threshold,
        nms_threshold=nms_threshold,
        bbox_margin=bbox_margin,
        box_smooth_alpha=box_smooth_alpha,
    ) if not use_cache else None

    common_cfg = dict(
        dataset_dir=dataset_dir,
        time_interval=time_interval,
        window_sec=window_sec,
        manifest_filename=manifest_filename,
        letterbox_size=letterbox_size,
        face_size=face_size,
        conf_threshold=conf_threshold,
        nms_threshold=nms_threshold,
        bbox_margin=bbox_margin,
        box_smooth_alpha=box_smooth_alpha,
        pipnet_backbone=pipnet_backbone,
        use_cache=use_cache,
        cache_dir=cache_dir,
        extractor=shared_extractor,
        device=device,
        extract_crops_only=extract_crops_only,
        selected_datasets=selected_datasets,
        use_visual_context=use_visual_context,
    )

    train_ds = LiteFatDataset(
        split="train",
        augment_train=augment_train,
        flip_prob=flip_prob,
        scale_jitter=scale_jitter,
        shift_jitter=shift_jitter,
        roll_deg=roll_deg,
        temporal_dropout=temporal_dropout,
        landmark_noise_std=landmark_noise_std,
        color_jitter=color_jitter,
        spatial_dropout=spatial_dropout,
        **common_cfg,
    )
    val_ds = LiteFatDataset(split="val", augment_train=False, **common_cfg)
    test_ds = LiteFatDataset(split="test", augment_train=False, **common_cfg)

    train_sampler = None
    if subject_balanced_sampling and len(train_ds) > 0:
        from collections import Counter
        sub_label_counts = Counter()
        label_counts = Counter()
        for s in train_ds.samples:
            sub = s.get("subject_id", "unknown")
            lbl = s.get("label", 0)
            sub_label_counts[(sub, lbl)] += 1
            label_counts[lbl] += 1

        sample_weights = []
        for s in train_ds.samples:
            sub = s.get("subject_id", "unknown")
            lbl = s.get("label", 0)
            cnt_sub = max(1, sub_label_counts[(sub, lbl)])
            cnt_lbl = max(1, label_counts[lbl])
            w = 1.0 / (math.sqrt(cnt_sub) * cnt_lbl)
            sample_weights.append(w)

        train_sampler = torch.utils.data.WeightedRandomSampler(
            weights=torch.as_tensor(sample_weights, dtype=torch.double),
            num_samples=len(sample_weights),
            replacement=True,
        )

    train_loader = DataLoader(
        train_ds, batch_size=batch_size,
        shuffle=(shuffle_train if train_sampler is None else False),
        sampler=train_sampler,
        num_workers=num_workers, pin_memory=pin_memory, collate_fn=collate_litefat
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=pin_memory, collate_fn=collate_litefat
    )
    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=pin_memory, collate_fn=collate_litefat
    )

    return train_loader, val_loader, test_loader

# =====================================================================

In [5]:
# =====================================================================
# File: model.py
# Dự án: Driver Guardian - Module LiteFat (IEEE/RSJ IROS 2025 Upgraded)
# Tác vụ: Định nghĩa kiến trúc LiteFat v2 giải quyết triệt để sụp đổ
#         đồ thị mốc mặt (Landmark Collapse) và hiện tượng học thuộc
#         lòng khuôn mặt (Shortcut Identity Overfitting) của MobileNetV3.
# =====================================================================
from __future__ import annotations

import math
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small


class KinematicFeatureExtractor(nn.Module):
    """
    Khối tiền xử lý chuẩn hóa động học 68 điểm mốc khuôn mặt (LiteFat v5):
    1. Chuẩn hóa hình thái đa cơ quan độc lập (Organ-Centered Local Coordinates):
       - Mắt & Lông mày: Quy chiếu về tâm khóe mắt, bảo toàn 100% độ mở mắt tĩnh/động (EAR).
       - Miệng: Quy chiếu về tâm khóe miệng, bảo toàn 100% độ mở miệng tĩnh/động (MAR).
       - Hàm & Mũi: Khử cấu trúc tĩnh (pos_dyn + 0.25*pos_norm) để ghi nhận tư thế đầu mà không học thuộc lòng xương sọ.
    2. Ổn định IOD tránh nổ thương số khi quay đầu sang ngang (Head Yaw).
    3. Đạo hàm vận tốc d(nose)/iod và gia tốc đúng chuẩn giải tích, loại bỏ triệt để gai giả ở t=1.
    4. Cân bằng năng lượng động học từng clip (Instance Kinematic Normalization) trước BatchNorm2d(6)
       để đồng bộ biên độ chuyển động giữa các bộ dữ liệu khác biệt FPS và độ rung cabin (UL-DD, SUST, VBDDD).
    """

    def __init__(
            self,
            nose_idx: int = 30,
            left_eye_idx: int = 36,
            right_eye_idx: int = 45,
            eps: float = 1e-2,
    ):
        super().__init__()
        self.nose_idx = nose_idx
        self.left_eye_idx = left_eye_idx
        self.right_eye_idx = right_eye_idx
        self.eps = eps

        # Chuẩn hóa phân phối riêng cho 6 kênh [x, y, dx, dy, d2x, d2y]
        self.channel_norm = nn.BatchNorm2d(6, affine=True)

    @staticmethod
    def build_temporal_mask(
            x: torch.Tensor,
            lengths: Optional[torch.Tensor] = None,
            eps: float = 1e-6,
    ) -> torch.Tensor:
        """
        Tạo binary float mask kích thước (B, 1, T, 1) biểu diễn các frame hợp lệ (1.0) và frame đệm (0.0).
        """
        B = x.size(0)
        T = x.size(2) if x.dim() == 4 else x.size(1)
        device = x.device
        if lengths is not None:
            t_idx = torch.arange(T, device=device).unsqueeze(0)  # (1, T)
            mask = (t_idx < lengths.to(device).unsqueeze(1)).float()  # (B, T)
            return mask.view(B, 1, T, 1)
        else:
            frame_activity = x[:, :2].abs().sum(dim=(1, 3), keepdim=True) if x.dim() == 4 and x.size(
                1) == 2 else x.abs().sum(dim=(-2, -1), keepdim=True)
            return (frame_activity > eps).float().view(B, 1, T, 1)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Input : (B, 2, T, V=68) - Tọa độ [0, 1] trong hệ tọa độ khung ảnh.
        mask  : (B, 1, T, 1)    - Float mask (1.0 = valid, 0.0 = padded/missing).
        Output: (B, 6, T, V=68) - [x_norm, y_norm, dx_norm, dy_norm, d2x_norm, d2y_norm].
        """
        if x.size(1) != 2:
            return x * mask if mask is not None else x

        B, _, T, V = x.shape

        # 1. Khoảng cách giải phẫu 2 mắt (Inter-Ocular Distance - IOD giữa điểm 36 và 45)
        p_left_eye = x[:, :, :, self.left_eye_idx: self.left_eye_idx + 1]  # (B, 2, T, 1)
        p_right_eye = x[:, :, :, self.right_eye_idx: self.right_eye_idx + 1]  # (B, 2, T, 1)
        raw_iod = torch.norm(p_right_eye - p_left_eye, p=2, dim=1, keepdim=True).clamp(min=self.eps)  # (B, 1, T, 1)

        # Ổn định IOD theo từng clip: kết hợp IOD tức thời với trung bình IOD của các frame hợp lệ
        # Khi tài xế quay đầu sang ngang (yaw), khoảng cách 2D co lại nhưng không làm nổ mẫu số
        if mask is not None:
            v_mask = mask.clamp(min=0.0, max=1.0)
            clip_iod = (raw_iod * v_mask).sum(dim=2, keepdim=True) / v_mask.sum(dim=2, keepdim=True).clamp(min=1.0)
        else:
            clip_iod = raw_iod.mean(dim=2, keepdim=True)
        clip_iod = clip_iod.clamp(min=self.eps)
        iod_scale = torch.maximum(0.5 * raw_iod + 0.5 * clip_iod, 0.65 * clip_iod).clamp(min=self.eps)

        # 2. Tọa độ chuẩn hóa theo Tâm Mũi (pos_norm) dùng cho biến dạng động lực học
        nose_pos = x[:, :, :, self.nose_idx: self.nose_idx + 1]  # (B, 2, T, 1)
        pos_norm = (x - nose_pos) / iod_scale  # (B, 2, T, V)
        pos_norm = torch.clamp(pos_norm, min=-8.0, max=8.0)

        # 3. Chuẩn hóa hình thái đa cơ quan độc lập (Organ-Centered Local Coordinates):
        # Mắt trái (36..41) & Lông mày trái (17..21): Gốc tại trung điểm khóe mắt trái (36 & 39)
        c_left_eye = 0.5 * (x[:, :, :, 36:37] + x[:, :, :, 39:40])
        # Mắt phải (42..47) & Lông mày phải (22..26): Gốc tại trung điểm khóe mắt phải (42 & 45)
        c_right_eye = 0.5 * (x[:, :, :, 42:43] + x[:, :, :, 45:46])
        # Miệng (48..67): Gốc tại trung điểm 2 khóe miệng (48 & 54)
        c_mouth = 0.5 * (x[:, :, :, 48:49] + x[:, :, :, 54:55])

        pos_feat = pos_norm.clone()
        # Vùng mắt & lông mày trái
        pos_feat[:, :, :, 17:22] = (x[:, :, :, 17:22] - c_left_eye) / iod_scale
        pos_feat[:, :, :, 36:42] = (x[:, :, :, 36:42] - c_left_eye) / iod_scale
        # Vùng mắt & lông mày phải
        pos_feat[:, :, :, 22:27] = (x[:, :, :, 22:27] - c_right_eye) / iod_scale
        pos_feat[:, :, :, 42:48] = (x[:, :, :, 42:48] - c_right_eye) / iod_scale
        # Vùng miệng
        pos_feat[:, :, :, 48:68] = (x[:, :, :, 48:68] - c_mouth) / iod_scale

        # Vùng Hàm (0..16) & Mũi (27..35): Trừ trung bình thời gian để bắt gật đầu/quay đầu mà không học thuộc lòng xương sọ
        jaw_nose_indices = list(range(0, 17)) + list(range(27, 36))
        pos_jn = pos_norm[:, :, :, jaw_nose_indices]
        if mask is not None:
            denom_jn = mask.sum(dim=2, keepdim=True).clamp(min=1.0)
            jn_mean = (pos_jn * mask).sum(dim=2, keepdim=True) / denom_jn
        else:
            jn_mean = pos_jn.mean(dim=2, keepdim=True)
        pos_feat[:, :, :, jaw_nose_indices] = (pos_jn - jn_mean) + 0.25 * pos_jn
        pos_feat = torch.clamp(pos_feat, min=-8.0, max=8.0)

        # 4. Tính Vận tốc (dx, dy) và Gia tốc (d2x, d2y) chuẩn hóa giải tích
        if T <= 1:
            vel_norm = torch.zeros_like(pos_norm)
            acc_norm = torch.zeros_like(pos_norm)
        else:
            # Vận tốc biến dạng cục bộ giữa các khung hình liên tiếp
            vel_local = torch.diff(pos_norm, n=1, dim=2)  # (B, 2, T-1, V)

            # Vận tốc tịnh tiến toàn đầu tại đỉnh tâm mũi (node 30):
            # Tính d(nose)/iod thay vì d(nose/iod) để triệt tiêu hoàn toàn lỗi nổ vận tốc do dao động mẫu số
            delta_nose = torch.diff(nose_pos, n=1, dim=2)  # (B, 2, T-1, 1)
            iod_mid = 0.5 * (iod_scale[:, :, 1:, :] + iod_scale[:, :, :-1, :])
            nose_vel = delta_nose / iod_mid.clamp(min=self.eps)
            nose_vel = torch.clamp(nose_vel, min=-4.0, max=4.0)
            vel_local = vel_local.clone()
            vel_local[:, :, :, self.nose_idx: self.nose_idx + 1] = nose_vel

            if mask is not None:
                pair_valid = mask[:, :, 1:, :] * mask[:, :, :-1, :]
                vel_local = vel_local * pair_valid

            # Gia tốc: Đạo hàm bậc 2 trên chuỗi vận tốc trước khi chèn số 0 (loại bỏ gai giả ở t=1)
            if T >= 3:
                acc_raw = torch.diff(vel_local, n=1, dim=2)  # (B, 2, T-2, V)
                if mask is not None:
                    triplet_valid = mask[:, :, 2:, :] * mask[:, :, 1:-1, :] * mask[:, :, :-2, :]
                    acc_raw = acc_raw * triplet_valid
                acc_raw = torch.clamp(acc_raw, min=-4.0, max=4.0)
                acc_norm = F.pad(acc_raw, (0, 0, 2, 0), mode="constant", value=0.0)  # (B, 2, T, V)
            else:
                acc_norm = torch.zeros_like(pos_norm)

            vel_norm = F.pad(vel_local, (0, 0, 1, 0), mode="constant", value=0.0)  # (B, 2, T, V)
            vel_norm = torch.clamp(vel_norm, min=-4.0, max=4.0)

        # 5. Cân bằng năng lượng động học từng clip (Instance Kinematic Normalization)
        # Giúp đồng bộ thang đo giữa UL-DD (camera 60fps tĩnh) và SUST/VBDDD (camera cabin rung)
        if mask is not None:
            m_valid = mask.clamp(min=0.0, max=1.0)
            n_valid = m_valid.sum(dim=(2, 3), keepdim=True).clamp(min=1.0) * float(V)
            vel_rms = torch.sqrt((vel_norm.pow(2) * m_valid).sum(dim=(2, 3), keepdim=True) / n_valid + 1e-6)
            acc_rms = torch.sqrt((acc_norm.pow(2) * m_valid).sum(dim=(2, 3), keepdim=True) / n_valid + 1e-6)
        else:
            vel_rms = torch.sqrt(vel_norm.pow(2).mean(dim=(2, 3), keepdim=True) + 1e-6)
            acc_rms = torch.sqrt(acc_norm.pow(2).mean(dim=(2, 3), keepdim=True) + 1e-6)

        # Chuẩn hóa mềm đưa RMS về dải tự nhiên ~0.035 IOD/frame
        vel_scale = torch.clamp(vel_rms, min=0.012, max=0.10) / 0.035
        acc_scale = torch.clamp(acc_rms, min=0.010, max=0.08) / 0.025
        vel_norm = vel_norm / vel_scale
        acc_norm = acc_norm / acc_scale

        out = torch.cat([pos_feat, vel_norm, acc_norm], dim=1)  # (B, 6, T, V)
        out = torch.nan_to_num(out, nan=0.0, posinf=4.0, neginf=-4.0)

        # 6. Chuẩn hóa kênh BatchNorm2d(6) kết hợp che mặt nạ chặt chẽ
        if mask is not None:
            out = out * mask
        out = self.channel_norm(out)
        if mask is not None:
            out = out * mask

        return out


class MobileNetV3ContextEncoder(nn.Module):
    """
    Trích xuất vector ngữ cảnh thị giác d in R^{D} cho mỗi khung hình khuôn mặt
    bằng xương sống MobileNetV3-Small.
    """

    def __init__(
            self,
            out_dim: int = 64,
            proj_hidden_dim: int = 192,
            pretrained: bool = True,
            dropout: float = 0.2,
            freeze_backbone: bool = False,
    ):
        super().__init__()
        weights = MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
        base_model = mobilenet_v3_small(weights=weights)

        self.features = base_model.features
        self.avgpool = base_model.avgpool

        self.projector = nn.Sequential(
            nn.Linear(576, proj_hidden_dim),
            nn.Hardswish(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(proj_hidden_dim, out_dim),
        )

        if freeze_backbone:
            self.freeze_backbone(freeze=True)

    def freeze_backbone(self, freeze: bool = True, freeze_projector: bool = False) -> None:
        """Bật/tắt đóng băng tham số của xương sống MobileNetV3-Small."""
        for param in self.features.parameters():
            param.requires_grad = not freeze
        for m in self.features.modules():
            if isinstance(m, (nn.BatchNorm2d, nn.SyncBatchNorm)):
                m.eval() if freeze else m.train(self.training)
        if freeze_projector:
            for param in self.projector.parameters():
                param.requires_grad = not freeze
        elif not freeze:
            for param in self.projector.parameters():
                param.requires_grad = True

    def unfreeze_backbone(self, unfreeze_last_n_blocks: Optional[int] = None) -> None:
        """
        Mở khóa đóng băng MobileNetV3-Small:
        - unfreeze_last_n_blocks = None: mở khóa toàn bộ features.
        - unfreeze_last_n_blocks = K: chỉ mở khóa K block tích chập cuối của features.
        """
        if unfreeze_last_n_blocks is None:
            self.freeze_backbone(freeze=False, freeze_projector=False)
            return

        # Khóa toàn bộ trước
        self.freeze_backbone(freeze=True, freeze_projector=False)
        total_blocks = len(self.features)
        start_idx = max(0, total_blocks - unfreeze_last_n_blocks)
        for i in range(start_idx, total_blocks):
            for param in self.features[i].parameters():
                param.requires_grad = True

    @property
    def is_backbone_frozen(self) -> bool:
        """Kiểm tra xem tất cả các tham số của MobileNetV3 features có đang bị đóng băng hay không."""
        return not any(p.requires_grad for p in self.features.parameters())

    def train(self, mode: bool = True):
        super().train(mode)
        if mode and self.is_backbone_frozen:
            for m in self.features.modules():
                if isinstance(m, (nn.BatchNorm2d, nn.SyncBatchNorm)):
                    m.eval()
        return self

    def forward(self, frame_tensor: torch.Tensor) -> torch.Tensor:
        feat = self.features(frame_tensor)
        feat = self.avgpool(feat)
        feat = torch.flatten(feat, 1)
        d = self.projector(feat)
        return d


class MultimodalFusionModule(nn.Module):
    """
    Dung hợp Đa phương thức giải quyết triệt để Rank-1 Bottleneck:
    1. Chiếu 6 kênh động học [x, y, dx, dy, d2x, d2y] lên không gian đặc trưng đa chiều D (Linear + LayerNorm + GELU).
    2. Cộng Node Positional Embedding [1, 1, 68, D] giúp phân biệt rõ vai trò giải phẫu của từng điểm mốc.
    3. Điều biến Gated FiLM modulation với vector ngữ cảnh thị giác d in R^{D}:
       H = H_kin * (1.0 + tanh(W_scale(d))) + W_shift(d)
    4. Tương thích hoàn hảo cả khi d=None (chế độ Fast Landmark-Only).
    """

    def __init__(
            self,
            num_features: int = 6,
            feature_dim: int = 64,
            num_nodes: int = 68,
            nose_idx: int = 30,
            left_eye_idx: int = 36,
            right_eye_idx: int = 45,
            eps: float = 1e-2,
    ):
        super().__init__()
        self.left_eye_idx = left_eye_idx
        self.right_eye_idx = right_eye_idx
        self.eps = eps
        self.feature_dim = feature_dim
        self.num_nodes = num_nodes

        # Khối tiền xử lý động học chuẩn hóa theo IOD
        self.kinematic = KinematicFeatureExtractor(
            nose_idx=nose_idx,
            left_eye_idx=left_eye_idx,
            right_eye_idx=right_eye_idx,
            eps=eps,
        )

        # Giữ lại w để tương thích backward compatibility với checkpoint cũ
        self.w = nn.Parameter(torch.ones(num_features, 1) / float(num_features))

        # Phép chiếu động học đa kênh 6D -> D
        self.kin_proj = nn.Sequential(
            nn.Linear(num_features, feature_dim),
            nn.LayerNorm(feature_dim),
            nn.GELU(),
        )

        # Vector nhúng vị trí đỉnh giải phẫu khả học cho 68 mốc mặt
        self.node_embed = nn.Parameter(torch.randn(1, 1, num_nodes, feature_dim) * 0.02)

        # Tầng điều biến FiLM từ vector ngữ cảnh thị giác d
        self.film_scale = nn.Linear(feature_dim, feature_dim)
        self.film_shift = nn.Linear(feature_dim, feature_dim)

    def detect_missing_mask(self, coords: torch.Tensor) -> torch.Tensor:
        p_left = coords[:, :, self.left_eye_idx, :]
        p_right = coords[:, :, self.right_eye_idx, :]
        raw_iod = torch.norm(p_right - p_left, p=2, dim=-1, keepdim=True).unsqueeze(-1)

        all_ones = (coords >= 0.999).all(dim=-1, keepdim=True).all(dim=-2, keepdim=True)
        all_zeros = (coords <= 0.001).all(dim=-1, keepdim=True).all(dim=-2, keepdim=True)
        is_missing = (raw_iod < 1e-2) | all_ones | all_zeros
        return is_missing

    def forward(
            self,
            coords: torch.Tensor,
            d: Optional[torch.Tensor] = None,
            mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Args:
            coords: Chuỗi tọa độ mốc [B, T, N, 2] (hoặc [B, T, N, 3] từ cache cũ).
            d: Vector ngữ cảnh thị giác [B, T, D] hoặc None (Landmark-Only).
            mask: Temporal float mask [B, 1, T, 1].
        Returns:
            X: Tensor đặc trưng đồ thị [B, D, N, T].
        """
        if coords.dim() == 4:
            if coords.size(-1) >= 2:
                coords_xy = coords[..., :2]
            elif coords.size(1) >= 2:
                coords_xy = coords[:, :2, :, :].permute(0, 2, 3, 1).contiguous()
            else:
                coords_xy = coords
        else:
            coords_xy = coords

        orig_dtype = coords.dtype
        dev = coords.device
        dev_type = dev.type if dev.type in ("cuda", "cpu") else "cpu"

        with torch.autocast(device_type=dev_type, enabled=False):
            coords_32 = coords_xy.float()

            is_missing = self.detect_missing_mask(coords_32)
            valid_mask = (~is_missing).permute(0, 3, 1, 2).float()

            if mask is not None:
                temporal_mask = mask.float() * valid_mask
                is_missing = (temporal_mask == 0.0).permute(0, 2, 3, 1)
            else:
                temporal_mask = valid_mask

            x_in = coords_32.permute(0, 3, 1, 2).contiguous()  # [B, 2, T, N]
            feat_kin = self.kinematic(x_in, mask=temporal_mask)  # [B, 6, T, N]
            C_kin = feat_kin.permute(0, 2, 3, 1).contiguous()  # [B, T, N, 6]

            # Chiếu đa kênh 6D -> D và cộng nhúng vị trí giải phẫu
            H_kin = self.kin_proj(C_kin) + self.node_embed  # [B, T, N, D]

            # Điều biến FiLM nếu có bối cảnh thị giác d
            if d is not None:
                d_32 = d.float()
                scale = torch.tanh(self.film_scale(d_32)).unsqueeze(2)  # [B, T, 1, D]
                shift = self.film_shift(d_32).unsqueeze(2)  # [B, T, 1, D]
                H = H_kin * (1.0 + scale) + shift
            else:
                H = H_kin

            H = torch.nan_to_num(H, nan=0.0)
            # Giữ các khung hình đệm (padding) bằng 0 tuyệt đối trước khi vào STGL
            H = H * temporal_mask.permute(0, 2, 3, 1)

            # Đưa về chuẩn đồ thị STGL: [B, D, N, T]
            X = H.permute(0, 3, 2, 1).contiguous()

        return X.to(orig_dtype)


class SelfAdaptiveAdjacency(nn.Module):
    """
    Ma trận kề kết hợp Cấu trúc Giải phẫu Khuôn mặt (A_prior) và Tự thích nghi (A_adp):
        A_total = A_prior + alpha * softmax(ReLU(E1 @ E2^T / sqrt(d)))
    """

    def __init__(
            self,
            num_nodes: int = 68,
            embed_dim: int = 16,
            vertical_pairs: Optional[List[Tuple[int, int]]] = None,
            cross_region_pairs: Optional[List[Tuple[int, int]]] = None,
    ):
        super().__init__()
        self.num_nodes = num_nodes
        self.embed_dim = embed_dim

        # Xây dựng ma trận kề giải phẫu A_prior chuẩn hóa bậc đối xứng
        A_prior = self._build_anatomical_prior(num_nodes, vertical_pairs, cross_region_pairs)
        self.register_buffer("A_prior", A_prior)

        # Tham số học ma trận kề thích nghi
        self.E1 = nn.Parameter(torch.randn(num_nodes, embed_dim) * 0.1)
        self.E2 = nn.Parameter(torch.randn(num_nodes, embed_dim) * 0.1)
        self.alpha = nn.Parameter(torch.tensor(0.5))

    @staticmethod
    def _build_anatomical_prior(
            num_nodes: int,
            vertical_pairs: Optional[List[Tuple[int, int]]],
            cross_region_pairs: Optional[List[Tuple[int, int]]],
    ) -> torch.Tensor:
        A = np.zeros((num_nodes, num_nodes), dtype=np.float32)

        # 1. Đường viền hàm dưới: 0..16
        for i in range(16):
            A[i, i + 1] = A[i + 1, i] = 1.0

        # 2. Lông mày trái (17..21) & phải (22..26)
        for i in range(17, 21):
            A[i, i + 1] = A[i + 1, i] = 1.0
        for i in range(22, 26):
            A[i, i + 1] = A[i + 1, i] = 1.0

        # 3. Sống mũi (27..30) & Cánh mũi (31..35)
        for i in range(27, 30):
            A[i, i + 1] = A[i + 1, i] = 1.0
        for i in range(31, 35):
            A[i, i + 1] = A[i + 1, i] = 1.0
        A[30, 33] = A[33, 30] = 1.0

        # 4. Mắt trái (36..41) & Mắt phải (42..47)
        for i in range(36, 41):
            A[i, i + 1] = A[i + 1, i] = 1.0
        A[41, 36] = A[36, 41] = 1.0
        for i in range(42, 47):
            A[i, i + 1] = A[i + 1, i] = 1.0
        A[47, 42] = A[42, 47] = 1.0

        # 5. Môi ngoài (48..59) & Môi trong (60..67)
        for i in range(48, 59):
            A[i, i + 1] = A[i + 1, i] = 1.0
        A[59, 48] = A[48, 59] = 1.0
        for i in range(60, 67):
            A[i, i + 1] = A[i + 1, i] = 1.0
        A[67, 60] = A[60, 67] = 1.0

        # 6. Các cặp dọc mở/nhắm mắt (EAR) & miệng (MAR)
        v_pairs = vertical_pairs or [
            (37, 41), (38, 40), (43, 47), (44, 46),
            (50, 58), (51, 57), (52, 56), (61, 67), (62, 66), (63, 65)
        ]
        for u, v in v_pairs:
            if u < num_nodes and v < num_nodes:
                A[u, v] = A[v, u] = 1.0

        # 7. Các cặp liên vùng khuôn mặt
        cr_pairs = cross_region_pairs or [
            (19, 37), (24, 44), (30, 39), (30, 42), (57, 8), (51, 33)
        ]
        for u, v in cr_pairs:
            if u < num_nodes and v < num_nodes:
                A[u, v] = A[v, u] = 1.0

        # Tự liên kết (Self-loops)
        A = A + np.eye(num_nodes, dtype=np.float32)

        # Chuẩn hóa bậc đối xứng: D^{-1/2} A D^{-1/2}
        degrees = np.sum(A, axis=1)
        d_inv_sqrt = np.power(np.maximum(degrees, 1e-8), -0.5)
        D_inv_sqrt = np.diag(d_inv_sqrt)
        A_norm = D_inv_sqrt @ A @ D_inv_sqrt

        return torch.from_numpy(A_norm).float()

    def forward(self) -> torch.Tensor:
        orig_dtype = self.E1.dtype
        scale = 1.0 / math.sqrt(self.embed_dim)
        E1_f32 = self.E1.float()
        E2_f32 = self.E2.float()
        spatial_weights = F.relu(torch.matmul(E1_f32, E2_f32.transpose(0, 1)) * scale)
        A_adp = F.softmax(spatial_weights, dim=-1)
        # Giới hạn alpha qua tanh để ma trận kề thích nghi không tăng trưởng vô hạn
        alpha_bounded = torch.tanh(self.alpha.float())
        A_total = self.A_prior.float() + alpha_bounded * A_adp
        return A_total.to(orig_dtype)


class SpatialGCNLayer(nn.Module):
    """
    Tầng Tích chập Đồ thị Không gian (GCN) không làm triệt tiêu gradient:
    Thay thế F.softmax bằng F.gelu(LayerNorm) kèm kết nối phần dư.
    """

    def __init__(
            self,
            in_dim: int = 64,
            hidden_dim: int = 64,
            out_dim: int = 64,
            dropout: float = 0.2,
    ):
        super().__init__()
        self.W0 = nn.Linear(in_dim, hidden_dim, bias=False)
        self.W1 = nn.Linear(hidden_dim, out_dim, bias=False)
        self.norm = nn.LayerNorm(out_dim)
        self.dropout = nn.Dropout(p=dropout)
        self.res_proj = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()

    def forward(self, x: torch.Tensor, A_adp: torch.Tensor) -> torch.Tensor:
        orig_dtype = x.dtype
        device_type = x.device.type
        # Bảo vệ số học: chạy phép nhân đồ thị và LayerNorm dưới float32 để chống tràn số float16 (max 65,504)
        with torch.amp.autocast(device_type, enabled=False):
            x_perm = x.float().permute(0, 3, 2, 1).contiguous()  # [B, T, N, D]
            A_f32 = A_adp.float()

            h1 = self.W0(x_perm)  # [B, T, N, H]
            # Nhân đồ thị chuẩn xác: đỉnh i tổng hợp đặc trưng từ các đỉnh láng giềng j theo A_adp[i, j]
            # torch.matmul(A_f32, h1) tự động broadcast qua [B, T], chạy bằng cuBLAS GEMM nhanh gấp 3-4 lần einsum
            h1 = torch.matmul(A_f32, h1)
            h1 = F.relu(h1)
            h1 = self.dropout(h1)

            h2 = self.W1(h1)  # [B, T, N, M]
            h2 = torch.matmul(A_f32, h2)
            z = F.gelu(self.norm(h2)) + self.res_proj(x_perm)

        return z.to(orig_dtype).permute(0, 3, 2, 1).contiguous()


class GatedDilatedCausalTCN(nn.Module):
    """
    Tầng Tích chập Thời gian Nhân quả Giãn nở có Cổng (Gated Dilated Causal TCN):
    h = tanh(Theta_1 * X) (x) sigmoid(Theta_2 * X).
    """

    def __init__(
            self,
            in_channels: int = 64,
            dilation_channels: int = 64,
            kernel_size: int = 2,
            dilation: int = 1,
    ):
        super().__init__()
        self.causal_padding = (kernel_size - 1) * dilation
        self.theta_1 = nn.Conv2d(
            in_channels,
            dilation_channels,
            kernel_size=(1, kernel_size),
            dilation=(1, dilation),
            bias=True,
        )
        self.theta_2 = nn.Conv2d(
            in_channels,
            dilation_channels,
            kernel_size=(1, kernel_size),
            dilation=(1, dilation),
            bias=True,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_pad = F.pad(x, (self.causal_padding, 0, 0, 0))
        tcn_a = torch.tanh(self.theta_1(x_pad))
        tcn_b = torch.sigmoid(self.theta_2(x_pad))
        return tcn_a * tcn_b


class STGLBlock(nn.Module):
    """
    Khối Không gian - Thời gian STGL Block kết hợp Gated TCN và Spatial GCN:
    Đường tắt skip_conv được tính SAU spatial_gcn để dòng gradient chảy xuyên suốt qua tầng đồ thị.
    """

    def __init__(
            self,
            residual_channels: int = 64,
            dilation_channels: int = 64,
            skip_channels: int = 128,
            kernel_size: int = 2,
            dilation: int = 1,
            dropout: float = 0.2,
    ):
        super().__init__()
        self.gated_tcn = GatedDilatedCausalTCN(
            in_channels=residual_channels,
            dilation_channels=dilation_channels,
            kernel_size=kernel_size,
            dilation=dilation,
        )
        self.spatial_gcn = SpatialGCNLayer(
            in_dim=dilation_channels,
            hidden_dim=residual_channels,
            out_dim=residual_channels,
            dropout=dropout,
        )
        # Skip connection lấy từ h_gcn để truyền gradient trực tiếp qua GCN
        self.skip_conv = nn.Conv2d(residual_channels, skip_channels, kernel_size=(1, 1))
        self.bn = nn.BatchNorm2d(residual_channels)
        self.res_scale = nn.Parameter(torch.ones(1))

    def forward(
            self,
            x: torch.Tensor,
            A_adp: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        residual = x
        h_tcn = self.gated_tcn(x)
        h_gcn = self.spatial_gcn(h_tcn, A_adp)
        skip = self.skip_conv(h_gcn)
        x_res = self.bn(self.res_scale * h_gcn + residual)
        return x_res, skip


class LiteFat(nn.Module):
    """
    Mô hình LiteFat v2 Hoàn chỉnh:
      - Hỗ trợ linh hoạt cả 2 chế độ: Đa phương thức (Multimodal) và Chỉ dùng Landmark (Landmark-Only).
      - Học ma trận kề thích nghi kết hợp với kề giải phẫu A_prior.
      - Attentive Node Pooling tập trung vào mắt và miệng thay vì trung bình đều.
    """

    def __init__(
            self,
            num_nodes: int = 68,
            feature_dim: int = 64,
            mobilenet_proj_hidden: int = 192,
            node_embed_dim: int = 16,
            residual_channels: int = 64,
            dilation_channels: int = 64,
            skip_channels: int = 128,
            end_channels: int = 256,
            num_classes: int = 2,
            num_blocks: int = 3,
            layers_per_block: int = 2,
            kernel_size: int = 2,
            pretrained_backbone: bool = True,
            dropout: float = 0.3,
            freeze_backbone: bool = False,
            vertical_pairs: Optional[List[Tuple[int, int]]] = None,
            cross_region_pairs: Optional[List[Tuple[int, int]]] = None,
    ):
        super().__init__()
        self.num_nodes = num_nodes
        self.feature_dim = feature_dim
        self.num_classes = num_classes
        self.num_blocks = num_blocks
        self.layers_per_block = layers_per_block
        self.kernel_size = kernel_size

        # 1. MobileNetV3-Small trích xuất ngữ cảnh thị giác từ ảnh mặt
        self.context_encoder = MobileNetV3ContextEncoder(
            out_dim=feature_dim,
            proj_hidden_dim=mobilenet_proj_hidden,
            pretrained=pretrained_backbone,
            dropout=dropout,
            freeze_backbone=freeze_backbone,
        )

        # 2. Dung hợp đa phương thức FiLM chống Rank-1 Bottleneck
        self.fusion = MultimodalFusionModule(
            num_features=6,
            feature_dim=feature_dim,
            num_nodes=num_nodes,
        )

        # 3. Ma trận kề giải phẫu + tự thích nghi
        self.adaptive_adj = SelfAdaptiveAdjacency(
            num_nodes=num_nodes,
            embed_dim=node_embed_dim,
            vertical_pairs=vertical_pairs,
            cross_region_pairs=cross_region_pairs,
        )

        # Tầng chiếu đầu vào đồ thị
        self.start_conv = nn.Conv2d(feature_dim, residual_channels, kernel_size=(1, 1))

        # 4. Các khối Không gian - Thời gian (STGL Blocks)
        self.stgl_blocks = nn.ModuleList()
        for _ in range(num_blocks):
            dilation = 1
            for _ in range(layers_per_block):
                self.stgl_blocks.append(
                    STGLBlock(
                        residual_channels=residual_channels,
                        dilation_channels=dilation_channels,
                        skip_channels=skip_channels,
                        kernel_size=kernel_size,
                        dilation=dilation,
                        dropout=dropout,
                    )
                )
                dilation *= 2

        # Đầu phân loại dự đoán cấp khung hình
        self.end_conv_1 = nn.Conv2d(skip_channels, end_channels, kernel_size=(1, 1))
        self.end_conv_2 = nn.Conv2d(end_channels, num_classes, kernel_size=(1, 1))

        # Attentive Node Pooling: Tự học trọng số chú ý trên đặc trưng 256D của 68 đỉnh khuôn mặt
        self.node_gate = nn.Sequential(
            nn.Conv2d(end_channels, 64, kernel_size=(1, 1)),
            nn.ReLU(),
            nn.Conv2d(64, 1, kernel_size=(1, 1)),
        )
        # Prior giải phẫu thiên vị vùng mắt (36..47) và vùng miệng (48..67)
        node_prior = torch.zeros(1, 1, num_nodes, 1)
        node_prior[:, :, 36:48, :] = 1.0  # Mắt
        node_prior[:, :, 48:68, :] = 1.0  # Miệng
        self.register_buffer("node_prior_bias", node_prior)

        # Attentive Temporal Pooling: Tự học trọng số chú ý từ vector đặc trưng khung hình (end_channels) thay vì class logits
        self.temp_norm = nn.LayerNorm(end_channels)
        self.temporal_gate = nn.Sequential(
            nn.Linear(end_channels, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )
        self.temp_scale = nn.Parameter(torch.tensor(2.0))

        # Đầu phân loại độc lập cấp đoạn video (Clip-Level Classifier):
        # Tách biệt không gian biểu diễn cấp đoạn và không gian dự đoán frame để không bị ảnh hưởng bởi bias âm của frame
        self.clip_classifier = nn.Sequential(
            nn.Linear(end_channels, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward_stgl_from_fused(
            self,
            X: torch.Tensor,
            lengths: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Chạy phần đồ thị STGL từ tensor đặc trưng X [B, D, N, T]."""
        A_adp = self.adaptive_adj()
        h = self.start_conv(X)

        B, _, _, T = X.shape
        t_mask = None
        if lengths is not None:
            t_idx = torch.arange(T, device=X.device).unsqueeze(0)  # [1, T]
            t_mask = (t_idx < lengths.to(X.device).unsqueeze(1)).float().view(B, 1, 1, T)

        skip_sum = None
        for block in self.stgl_blocks:
            h, skip = block(h, A_adp)
            if t_mask is not None:
                h = h * t_mask
                skip = skip * t_mask
            skip_sum = skip if skip_sum is None else (skip_sum + skip)

        out = F.relu(skip_sum)
        h_end = F.relu(self.end_conv_1(out))  # [B, end_channels, N, T]

        # Attentive Spatial Pooling trên N=68 đỉnh dựa trên không gian đặc trưng 256D và prior giải phẫu
        node_attn = self.node_gate(h_end) + self.node_prior_bias  # [B, 1, N, T]
        gate = F.softmax(node_attn, dim=2)  # [B, 1, N, T]

        # Đặc trưng cấp khung hình sau khi gom tụ các đỉnh [B, end_channels, T]
        pooled_node_feat = (h_end * gate).sum(dim=2)  # [B, end_channels, T]
        frame_feat = pooled_node_feat.permute(0, 2, 1).contiguous()  # [B, T, end_channels]

        # Dự đoán logits cấp khung hình [B, T, num_classes]
        frame_logits = self.end_conv_2(pooled_node_feat.unsqueeze(2)).squeeze(2).permute(0, 2, 1).contiguous()

        # Multi-Scale Event MIL Pooling cho Clip Logits
        feat_norm = self.temp_norm(frame_feat)
        temp_scores = self.temporal_gate(feat_norm) * self.temp_scale  # [B, T, 1]

        if lengths is not None:
            valid_bool = (torch.arange(T, device=frame_logits.device).unsqueeze(0) < lengths.to(
                frame_logits.device).unsqueeze(1)).unsqueeze(-1)
            valid_mask = valid_bool.type_as(frame_logits)
            denom = valid_mask.sum(dim=1).clamp(min=1.0)

            # 1. Trọng số chú ý sắc nét (Sharp Attention)
            masked_scores = temp_scores.masked_fill(~valid_bool, -1e4)
            temp_weights = F.softmax(masked_scores, dim=1) * valid_mask
            temp_weights = temp_weights / temp_weights.sum(dim=1, keepdim=True).clamp(min=1e-6)

            # 2. Gộp đặc trưng đa nhánh: Mean + Sharp Attention + Top-K Event
            feat_mean = (frame_feat * valid_mask).sum(dim=1) / denom  # [B, end_channels]
            feat_attn = (frame_feat * temp_weights).sum(dim=1)  # [B, end_channels]

            # Top-K event pooling (lấy top 25% frames có điểm chú ý sự kiện cao nhất)
            k_val = max(1, int(round(T * 0.25)))
            scores_2d = masked_scores.squeeze(-1)  # [B, T]
            topk_idx = torch.topk(scores_2d, k=k_val, dim=1).indices  # [B, K]
            batch_indices = torch.arange(B, device=frame_feat.device).unsqueeze(1).expand(-1, k_val)
            feat_topk = frame_feat[batch_indices, topk_idx, :].mean(dim=1)  # [B, end_channels]

            pooled_clip_feat = 0.50 * feat_attn + 0.30 * feat_topk + 0.20 * feat_mean
            clip_head_logits = self.clip_classifier(pooled_clip_feat)  # [B, num_classes]

            # Kết hợp với attentive frame logits
            attn_frame_logits = (frame_logits * temp_weights).sum(dim=1)
            clip_logits = 0.60 * clip_head_logits + 0.40 * attn_frame_logits
        else:
            temp_weights = F.softmax(temp_scores, dim=1)
            feat_mean = frame_feat.mean(dim=1)
            feat_attn = (frame_feat * temp_weights).sum(dim=1)

            k_val = max(1, int(round(T * 0.25)))
            scores_2d = temp_scores.squeeze(-1)
            topk_idx = torch.topk(scores_2d, k=k_val, dim=1).indices
            batch_indices = torch.arange(B, device=frame_feat.device).unsqueeze(1).expand(-1, k_val)
            feat_topk = frame_feat[batch_indices, topk_idx, :].mean(dim=1)

            pooled_clip_feat = 0.50 * feat_attn + 0.30 * feat_topk + 0.20 * feat_mean
            clip_head_logits = self.clip_classifier(pooled_clip_feat)
            attn_frame_logits = (frame_logits * temp_weights).sum(dim=1)
            clip_logits = 0.60 * clip_head_logits + 0.40 * attn_frame_logits

        return frame_logits, clip_logits

    def forward(
            self,
            video_frames: Optional[torch.Tensor] = None,
            landmarks: Optional[torch.Tensor] = None,
            lengths: Optional[torch.Tensor] = None,
            mask: Optional[torch.Tensor] = None,
            return_clip_logits: bool = True,
    ) -> Union[torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        """
        Chuẩn hóa nạp duy nhất một kiểu:
            model(video_frames=..., landmarks=..., lengths=..., return_clip_logits=...)

        - Khi video_frames is not None và video_frames.numel() > 0:
            Trích xuất vector ngữ cảnh d [B, T, D] từ MobileNetV3 Context Encoder.
        - Khi video_frames is None:
            Chạy chế độ Fast Landmark-Only (d = None, chỉ sử dụng nhánh động học mốc mặt).
        """
        if landmarks is None:
            raise ValueError("Tham số 'landmarks' [B, T, N, 2] bắt buộc phải được cung cấp!")

        d = None
        if video_frames is not None and video_frames.numel() > 0:
            B, T, C_img, H, W = video_frames.shape
            frames_flat = video_frames.view(B * T, C_img, H, W)
            d_flat = self.context_encoder(frames_flat)
            d = d_flat.view(B, T, self.feature_dim)

        if mask is None and lengths is not None:
            mask = KinematicFeatureExtractor.build_temporal_mask(
                landmarks.permute(0, 3, 1, 2) if landmarks.dim() == 4 and landmarks.size(-1) == 2 else landmarks,
                lengths=lengths,
            )

        X = self.fusion(landmarks, d=d, mask=mask)
        frame_logits, clip_logits = self.forward_stgl_from_fused(X, lengths=lengths)

        if return_clip_logits:
            return frame_logits, clip_logits
        return frame_logits

    @torch.no_grad()
    def forward_asymmetric_step(
            self,
            landmark_window: torch.Tensor,
            d_history: torch.Tensor,
            current_frame: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        if current_frame is not None:
            d_new = self.context_encoder(current_frame).unsqueeze(1)
        else:
            d_new = d_history[:, -1:, :]

        d_updated = torch.cat([d_history[:, 1:, :], d_new], dim=1)
        X = self.fusion(landmark_window, d=d_updated)
        _, clip_logits = self.forward_stgl_from_fused(X)
        probs = F.softmax(clip_logits, dim=-1)
        return probs, d_updated

    def freeze_backbone(self, freeze: bool = True, freeze_projector: bool = False) -> None:
        self.context_encoder.freeze_backbone(freeze=freeze, freeze_projector=freeze_projector)

    def unfreeze_backbone(self, unfreeze_last_n_blocks: Optional[int] = None) -> None:
        self.context_encoder.unfreeze_backbone(unfreeze_last_n_blocks=unfreeze_last_n_blocks)

    def set_backbone_trainable(self, trainable: bool, unfreeze_last_n_blocks: Optional[int] = 3) -> None:
        """Đóng/mở khóa backbone MobileNetV3 theo epoch huấn luyện."""
        if not trainable:
            self.freeze_backbone(freeze=True, freeze_projector=False)
        else:
            self.unfreeze_backbone(unfreeze_last_n_blocks=unfreeze_last_n_blocks)

    @property
    def is_backbone_frozen(self) -> bool:
        return self.context_encoder.is_backbone_frozen

    def get_trainable_params_count(self) -> Dict[str, Any]:
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        backbone_total = sum(p.numel() for p in self.context_encoder.features.parameters())
        backbone_trainable = sum(p.numel() for p in self.context_encoder.features.parameters() if p.requires_grad)
        return {
            "total_params": total,
            "trainable_params": trainable,
            "frozen_params": total - trainable,
            "backbone_total": backbone_total,
            "backbone_trainable": backbone_trainable,
            "is_backbone_frozen": self.is_backbone_frozen,
        }


In [6]:
# =====================================================================
# File: loss.py
# Dự án: Driver Guardian - Module LiteFat
# Tác vụ: Định nghĩa hàm mục tiêu huấn luyện LiteFatLoss kết hợp
#         mất mát cấp đoạn video (Clip CrossEntropy) và mất mát
#         từng khung hình có trọng số thời gian nhân quả (Causal Frame Loss).
# =====================================================================
from __future__ import annotations

from typing import Optional, Union

import torch
import torch.nn as nn
import torch.nn.functional as F


class LiteFatLoss(nn.Module):
    """
    Hàm mất mát hỗn hợp (Hybrid Causal Loss & Multiple Instance Learning - MIL) cho mô hình LiteFat:
    L_total = clip_loss_weight * L_clip + frame_loss_weight * L_frame

    Trong đó:
      - L_clip: CrossEntropyLoss với label_smoothing trên clip_logits [B, num_classes].
      - L_frame: Multiple Instance Learning Frame Loss:
        + Với clip Alert (0): Giám sát 100% frame hợp lệ với nhãn 0.
        + Với clip Drowsy (1): Chỉ giám sát Top-K (45%) frame có xu hướng buồn ngủ cao nhất,
          tránh ép nhãn 1 cho các frame tỉnh táo bình thường trong clip buồn ngủ (ngăn ngừa kéo tụt bias âm).
        + Kết hợp trọng số nhân quả tăng dần w_t để TCN tích lũy đủ ngữ cảnh.
    """

    def __init__(
            self,
            clip_loss_weight: float = 1.0,
            frame_loss_weight: float = 0.3,
            label_smoothing: float = 0.1,
            topk_frame_ratio: float = 0.45,
    ):
        super().__init__()
        self.clip_loss_weight = float(clip_loss_weight)
        self.frame_loss_weight = float(frame_loss_weight)
        self.topk_frame_ratio = float(topk_frame_ratio)
        self.ce = nn.CrossEntropyLoss(reduction="mean", label_smoothing=label_smoothing)
        self.frame_ce_raw = nn.CrossEntropyLoss(reduction="none", label_smoothing=label_smoothing)

    def forward(
            self,
            frame_logits: torch.Tensor,
            targets: torch.Tensor,
            clip_logits: Optional[torch.Tensor] = None,
            lengths: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Args:
            frame_logits: Logits từng khung hình [B, T, num_classes].
            targets: Nhãn nhị phân / đa lớp [B] (hoặc [B, T]).
            clip_logits: Logits gộp toàn đoạn [B, num_classes] (tùy chọn).
            lengths: Độ dài frame thực tế của mỗi mẫu trong batch [B].
        """
        B, T, num_classes = frame_logits.shape
        device = frame_logits.device

        # Chuẩn hóa nhãn targets
        if targets.dim() == 1:
            clip_targets = targets
            frame_targets = targets.unsqueeze(1).expand(B, T)
        else:
            clip_targets = targets[:, -1]
            frame_targets = targets

        # 1. Tính Clip Loss
        if clip_logits is not None:
            loss_clip = self.ce(clip_logits, clip_targets)
        else:
            # Fallback nếu không truyền clip_logits: lấy trung bình frame_logits
            if lengths is not None:
                mask_t = (torch.arange(T, device=device).unsqueeze(0) < lengths.to(device).unsqueeze(1)).unsqueeze(
                    -1).float()
                avg_clip_logits = (frame_logits * mask_t).sum(dim=1) / mask_t.sum(dim=1).clamp(min=1.0)
            else:
                avg_clip_logits = frame_logits.mean(dim=1)
            loss_clip = self.ce(avg_clip_logits, clip_targets)

        # 2. Tính Causal Time-Weighted Top-K MIL Frame Loss
        if self.frame_loss_weight > 0:
            # Trọng số thời gian nhân quả mượt mà: tỷ lệ 0.5 ở frame đầu đến 1.0 ở frame cuối
            base_weights = 0.5 + 0.5 * torch.linspace(0.0, 1.0, T, device=device, dtype=torch.float32)
            time_weights = base_weights / base_weights.sum()  # [T]
            weight_matrix = time_weights.unsqueeze(0).expand(B, T).clone()  # [B, T]

            # Mặt nạ loại bỏ frame đệm (padding)
            if lengths is not None:
                valid_mask = torch.arange(T, device=device).unsqueeze(0) < lengths.to(device).unsqueeze(1)
            else:
                valid_mask = torch.ones((B, T), dtype=torch.bool, device=device)
            weight_matrix = weight_matrix * valid_mask.float()

            # Multiple Instance Learning (MIL) Frame Selection:
            # - Với mẫu Alert (0): toàn bộ valid frames là tỉnh táo -> giám sát 100% valid frames.
            # - Với mẫu Drowsy (1): chỉ giám sát Top-K (45%) frames có xác suất buồn ngủ cao nhất.
            drowsy_scores = frame_logits[:, :, 1] - frame_logits[:, :, 0]  # [B, T]
            drowsy_scores = drowsy_scores.masked_fill(~valid_mask, -1e4)

            mil_mask = valid_mask.clone()
            for b in range(B):
                if clip_targets[b] == 1:
                    t_len = int(lengths[b].item()) if lengths is not None else T
                    k_b = max(1, int(round(t_len * self.topk_frame_ratio)))
                    if k_b < t_len:
                        _, topk_idx = torch.topk(drowsy_scores[b, :t_len], k=k_b, dim=0)
                        sample_mask = torch.zeros(T, dtype=torch.bool, device=device)
                        sample_mask[topk_idx] = True
                        mil_mask[b] = sample_mask

            weight_matrix = weight_matrix * mil_mask.float()

            # Chuẩn hóa trọng số trên từng sample để mỗi video đóng góp đồng đều
            sample_weight_sum = weight_matrix.sum(dim=1, keepdim=True).clamp(min=1e-6)
            norm_weights = weight_matrix / sample_weight_sum  # [B, T]

            # Tính CE riêng cho từng frame [B * T] -> [B, T]
            ce_per_frame = self.frame_ce_raw(
                frame_logits.reshape(B * T, num_classes),
                frame_targets.reshape(B * T)
            ).reshape(B, T)

            loss_frame = (ce_per_frame * norm_weights).sum(dim=1).mean()
        else:
            loss_frame = torch.tensor(0.0, device=device)

        total_loss = self.clip_loss_weight * loss_clip + self.frame_loss_weight * loss_frame
        return total_loss


In [7]:
# =====================================================================
# File: train.py
# Dự án: Driver Guardian - Module LiteFat (IEEE/RSJ IROS 2025 Upgraded)
# Tác vụ: Lớp huấn luyện LiteFatTrainer thực thi pipeline huấn luyện
#         theo cấu hình TrainConfig trong config.py.
# =====================================================================
from __future__ import annotations

import json
import math
import os
import sys
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

try:
    from config import TrainConfig
    from dataset import build_litefat_dataloaders
    from loss import LiteFatLoss
    from model import LiteFat
    from ultis import atomic_save, compute_metrics, configure_logging, seed_everything
except ImportError:
    pass


class LiteFatTrainer:
    """
    Trình quản lý và thực thi huấn luyện mô hình LiteFat:
      - Hỗ trợ Differential Learning Rate (tốc độ học riêng cho Backbone và GCN/TCN).
      - Đóng băng Backbone theo giai đoạn (Staged Backbone Freezing).
      - Linear Warmup + Cosine Annealing Learning Rate.
      - Tăng cường dữ liệu không gian - thời gian đối xứng trên Landmark & Video Frames.
      - Lưu checkpoint an toàn (Atomic Save) và theo dõi qua TensorBoard.
    """

    def __init__(self, config: Optional[TrainConfig] = None):
        self.cfg = config if config is not None else TrainConfig()

        # 1. Thiết lập hạt giống ngẫu nhiên để tái lập kết quả
        seed_everything(self.cfg.seed)

        # 2. Thiết lập thư mục lưu trữ và Logging
        self.save_dir = Path(self.cfg.checkpoint_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.logger = configure_logging(self.save_dir)

        # Lưu lại cấu hình JSON của phiên huấn luyện
        self.cfg.save_json(self.save_dir / "config.json")

        # Khởi tạo TensorBoard SummaryWriter
        log_path = Path(self.cfg.log_dir)
        log_path.mkdir(parents=True, exist_ok=True)
        self.writer = SummaryWriter(log_dir=str(log_path))

        # 3. Khởi tạo Thiết bị tính toán (CUDA GPU / CPU)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.logger.info("=" * 70)
        self.logger.info(f"KHỞI TẠO LITEFAT TRAINER (Thiết bị: {self.device})")
        self.logger.info(f"Chế độ Bối cảnh Thị giác (use_visual_context): {self.cfg.use_visual_context}")
        self.logger.info(
            f"Thời gian lấy mẫu (time_interval): {self.cfg.time_interval}s | Cửa sổ: {self.cfg.window_sec}s")
        self.logger.info(f"Bộ dữ liệu chọn lọc: {self.cfg.selected_datasets or 'Tất cả'}")
        self.logger.info("=" * 70)

        # 4. Xây dựng DataLoaders (Train, Val, Test)
        self.logger.info("Đang khởi tạo DataLoaders...")
        self.train_loader, self.val_loader, self.test_loader = build_litefat_dataloaders(
            dataset_dir=self.cfg.dataset_dir,
            batch_size=self.cfg.batch_size,
            num_workers=self.cfg.num_workers,
            time_interval=self.cfg.time_interval,
            window_sec=self.cfg.window_sec,
            letterbox_size=self.cfg.letterbox_size,
            face_size=self.cfg.face_size,
            manifest_filename=self.cfg.manifest_filename,
            use_cache=self.cfg.use_cache,
            cache_dir=self.cfg.cache_dir,
            pin_memory=self.cfg.pin_memory,
            shuffle_train=True,
            device=str(self.device),
            extract_crops_only=(not self.cfg.use_visual_context),
            selected_datasets=self.cfg.selected_datasets,
            use_visual_context=self.cfg.use_visual_context,
            augment_train=self.cfg.augment_train,
            flip_prob=self.cfg.flip_prob,
            scale_jitter=self.cfg.scale_jitter,
            shift_jitter=self.cfg.shift_jitter,
            roll_deg=getattr(self.cfg, "roll_deg", 12.0),
            temporal_dropout=getattr(self.cfg, "temporal_dropout", 0.05),
            landmark_noise_std=self.cfg.landmark_noise_std,
            color_jitter=self.cfg.color_jitter,
            spatial_dropout=self.cfg.spatial_dropout,
            subject_balanced_sampling=getattr(self.cfg, "subject_balanced_sampling", True),
        )
        self.logger.info(
            f"Kích thước tập dữ liệu: Train={len(self.train_loader.dataset)} | "
            f"Val={len(self.val_loader.dataset)} | Test={len(self.test_loader.dataset)}"
        )

        # 5. Khởi tạo Mô hình LiteFat
        self.logger.info("Đang khởi tạo mô hình LiteFat...")
        self.model = LiteFat(
            num_nodes=self.cfg.num_nodes,
            feature_dim=self.cfg.feature_dim,
            mobilenet_proj_hidden=self.cfg.mobilenet_proj_hidden,
            node_embed_dim=self.cfg.node_embed_dim,
            residual_channels=self.cfg.residual_channels,
            dilation_channels=self.cfg.dilation_channels,
            skip_channels=self.cfg.skip_channels,
            end_channels=self.cfg.end_channels,
            num_classes=self.cfg.num_classes,
            num_blocks=self.cfg.num_blocks,
            layers_per_block=self.cfg.layers_per_block,
            kernel_size=self.cfg.kernel_size,
            pretrained_backbone=self.cfg.pretrained_backbone,
            dropout=self.cfg.dropout,
            freeze_backbone=(self.cfg.freeze_backbone_epochs > 0),
            vertical_pairs=self.cfg.vertical_pairs,
            cross_region_pairs=self.cfg.cross_region_pairs,
        ).to(self.device)

        stats = self.model.get_trainable_params_count()
        self.logger.info(
            f"Tổng số tham số: {stats['total_params']:,} | Khả học: {stats['trainable_params']:,} "
            f"(MobileNetV3: {stats['backbone_trainable']:,} / {stats['backbone_total']:,})"
        )

        # 6. Khởi tạo Hàm mất mát LiteFatLoss
        self.criterion = LiteFatLoss(
            clip_loss_weight=self.cfg.clip_loss_weight,
            frame_loss_weight=self.cfg.frame_loss_weight,
            label_smoothing=self.cfg.label_smoothing,
        ).to(self.device)

        # Các biến theo dõi thực nghiệm
        self.start_epoch = 1
        self.best_val_f1 = -1.0
        self.best_val_score = -1.0
        self.best_epoch = -1
        self.history: List[Dict[str, Any]] = []
        self.pretrained_param_names: set[str] = set()

        # Nạp trọng số tiền huấn luyện (Warm-start) nếu được yêu cầu trước khi tạo Optimizer
        if getattr(self.cfg, "pretrained_from", None) and not self.cfg.resume:
            self.load_pretrained_weights(self.cfg.pretrained_from)

        # 7. Thiết lập Bộ tối ưu hóa Differential Learning Rate (AdamW)
        self.optimizer = self._build_optimizer()

        # 8. Thiết lập Bộ điều chỉnh Tốc độ học (Warmup + CosineAnnealingLR)
        self.scheduler = self._build_scheduler()

        # 9. Bộ khuếch đại độ chính xác hỗn hợp Automatic Mixed Precision (AMP)
        amp_enabled = self.cfg.use_amp and self.device.type == "cuda"
        self.scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

        # Tự động khôi phục nếu cfg.resume được cấu hình
        if self.cfg.resume:
            self.load_checkpoint(self.cfg.resume)

    def _build_optimizer(self) -> torch.optim.Optimizer:
        """Thiết lập tốc độ học riêng biệt (Differential LR) cho Backbone, Pretrained STGL và New STGL."""
        backbone_params = []
        matched_graph_params = []
        new_graph_params = []

        pretrained_scale = float(getattr(self.cfg, "pretrained_lr_scale", 1.0))
        has_pretrained = bool(self.pretrained_param_names) and (pretrained_scale < 1.0)

        for name, param in self.model.named_parameters():
            if not param.requires_grad:
                continue
            if "context_encoder.features" in name:
                backbone_params.append(param)
            elif has_pretrained and name in self.pretrained_param_names:
                matched_graph_params.append(param)
            else:
                new_graph_params.append(param)

        param_groups = []
        if matched_graph_params:
            param_groups.append({
                "params": matched_graph_params,
                "lr": self.cfg.lr * pretrained_scale,
                "weight_decay": self.cfg.weight_decay,
            })
            self.logger.info(
                f"Đã gán Warm-start LR={self.cfg.lr * pretrained_scale:.2e} (scale={pretrained_scale}) cho {len(matched_graph_params)} tham số pretrained."
            )
        if new_graph_params:
            param_groups.append({
                "params": new_graph_params,
                "lr": self.cfg.lr,
                "weight_decay": self.cfg.weight_decay,
            })
        if backbone_params and self.cfg.use_visual_context:
            param_groups.append({
                "params": backbone_params,
                "lr": self.cfg.backbone_lr,
                "weight_decay": self.cfg.weight_decay,
            })

        optimizer = torch.optim.AdamW(param_groups)
        return optimizer

    def _safe_load_optimizer(self, opt_state_dict: Dict[str, Any]) -> None:
        """Nạp an toàn optimizer state dict, xử lý tương thích khi số lượng param_groups khác nhau."""
        saved_groups = len(opt_state_dict.get("param_groups", []))
        curr_groups = len(self.optimizer.param_groups)

        if saved_groups == curr_groups:
            self.optimizer.load_state_dict(opt_state_dict)
        elif saved_groups == 1 and curr_groups == 2:
            self.logger.warning(
                "Checkpoint optimizer state chỉ có 1 nhóm tham số (Graph/Head). "
                "Tự động ánh xạ nhóm 0 và khởi tạo nhóm Backbone ở trạng thái mới."
            )
            adapted_dict = {
                "state": opt_state_dict.get("state", {}),
                "param_groups": [
                    opt_state_dict["param_groups"][0],
                    self.optimizer.state_dict()["param_groups"][1],
                ],
            }
            self.optimizer.load_state_dict(adapted_dict)
        else:
            self.logger.warning(
                f"Không tương thích số lượng param_groups (checkpoint: {saved_groups}, hiện tại: {curr_groups}). "
                "Bỏ qua khôi phục optimizer state."
            )

    def load_checkpoint(self, checkpoint_path: str | Path) -> None:
        """Nạp toàn bộ trạng thái từ tệp checkpoint để tiếp tục huấn luyện."""
        ckpt_path = Path(checkpoint_path)
        if not ckpt_path.is_file():
            alt_path = self.save_dir / checkpoint_path
            if alt_path.is_file():
                ckpt_path = alt_path
            else:
                available_ckpts = list(self.save_dir.glob("*.pt"))
                available_names = [c.name for c in available_ckpts]
                raise FileNotFoundError(
                    f"Không tìm thấy tệp checkpoint tại '{checkpoint_path}' hoặc '{alt_path}'. "
                    f"Các checkpoint hiện có trong {self.save_dir}: {available_names}"
                )

        self.logger.info(f"Đang nạp checkpoint để tiếp tục huấn luyện từ: {ckpt_path}")
        checkpoint = torch.load(ckpt_path, map_location=self.device, weights_only=False)

        # 1. Khôi phục trọng số mô hình
        if "model_state_dict" in checkpoint:
            self.model.load_state_dict(checkpoint["model_state_dict"])
        else:
            self.logger.warning("Checkpoint không chứa 'model_state_dict'!")

        # 2. Khôi phục Epochs trước để tính đúng remaining_epochs cho Scheduler
        ckpt_epoch = checkpoint.get("epoch", 0)
        self.start_epoch = ckpt_epoch + 1

        # 3. Khôi phục Optimizer & Scheduler & GradScaler
        if "optimizer_state_dict" in checkpoint:
            self._safe_load_optimizer(checkpoint["optimizer_state_dict"])
        if "scheduler_state_dict" in checkpoint:
            saved_cfg_epochs = checkpoint.get("config", {}).get("epochs", None)
            if saved_cfg_epochs is not None and saved_cfg_epochs != self.cfg.epochs:
                self.logger.warning(
                    f"Phát hiện thay đổi tổng số epochs ({saved_cfg_epochs} -> {self.cfg.epochs}). "
                    f"Tự động tái lập lịch trình Cosine mượt từ Epoch {self.start_epoch} đến Epoch {self.cfg.epochs}."
                )
                remaining_epochs = max(1, self.cfg.epochs - self.start_epoch + 1)

                def resume_lr_lambda(ep_offset: int) -> float:
                    progress = float(ep_offset) / float(max(1, remaining_epochs))
                    return 0.5 * (1.0 + math.cos(math.pi * progress))

                self.scheduler = torch.optim.lr_scheduler.LambdaLR(self.optimizer, lr_lambda=resume_lr_lambda)
            else:
                self.scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        if "scaler_state_dict" in checkpoint and self.scaler.is_enabled():
            self.scaler.load_state_dict(checkpoint["scaler_state_dict"])

        # 4. Khôi phục Best Metrics

        if "best_epoch" in checkpoint:
            self.best_epoch = checkpoint.get("best_epoch", -1)
            self.best_val_f1 = checkpoint.get("best_val_f1", -1.0)
            self.best_val_score = checkpoint.get("best_val_score", self.best_val_f1)
        else:
            # Dự phòng từ best_metrics.json hoặc metrics của checkpoint
            best_json_path = self.save_dir / "best_metrics.json"
            if best_json_path.is_file():
                try:
                    with open(best_json_path, "r", encoding="utf-8") as f:
                        data = json.load(f)
                    if data.get("best_epoch", 0) <= ckpt_epoch:
                        self.best_epoch = data["best_epoch"]
                        self.best_val_f1 = data.get("metrics", {}).get("f1", -1.0)
                        self.best_val_score = data.get("metrics", {}).get("val_score", self.best_val_f1)
                except Exception:
                    pass
            if self.best_val_f1 < 0 and "val_metrics" in checkpoint:
                self.best_val_f1 = checkpoint["val_metrics"].get("f1", -1.0)
                self.best_val_score = checkpoint["val_metrics"].get("val_score", self.best_val_f1)
                self.best_epoch = ckpt_epoch

        # 4. Đồng bộ trạng thái đóng/mở băng backbone theo start_epoch
        if self.start_epoch <= self.cfg.freeze_backbone_epochs:
            self.model.set_backbone_trainable(False)
        else:
            self.model.set_backbone_trainable(True, unfreeze_last_n_blocks=3)

        self.logger.info("=" * 70)
        self.logger.info(f"===> KHÔI PHỤC CHECKPOINT THÀNH CÔNG TỪ: {ckpt_path.name}")
        self.logger.info(
            f"===> Đã hoàn thành Epoch {ckpt_epoch}. "
            f"Sẽ tiếp tục huấn luyện từ Epoch {self.start_epoch} đến Epoch {self.cfg.epochs}."
        )
        self.logger.info(
            f"===> Kỷ lục Best Val F1 hiện tại: {self.best_val_f1:.4f} (tại Epoch {self.best_epoch})"
        )
        self.logger.info("=" * 70)

    def load_pretrained_weights(self, pretrained_path: str | Path) -> None:
        """Nạp trọng số từng phần (Warm-start) từ checkpoint trước đó cho các tầng khớp kích thước, hỗ trợ ánh xạ theo block."""
        p = Path(pretrained_path)
        if not p.is_file():
            self.logger.warning(f"Không tìm thấy file pretrained_from tại '{pretrained_path}'. Bỏ qua nạp warm-start.")
            return

        self.logger.info(f"Đang nạp trọng số tiền huấn luyện (Warm-start) từ: {p}")
        checkpoint = torch.load(p, map_location=self.device, weights_only=False)
        saved_dict = checkpoint.get("model_state_dict", checkpoint)
        curr_dict = self.model.state_dict()

        # Xác định cấu trúc số tầng STGL của checkpoint cũ vs mô hình hiện tại
        saved_stgl_indices = sorted(list(set(int(k.split(".")[1]) for k in saved_dict if k.startswith("stgl_blocks."))))
        curr_stgl_indices = sorted(list(set(int(k.split(".")[1]) for k in curr_dict if k.startswith("stgl_blocks."))))

        num_blocks = getattr(self.model, "num_blocks", 3)
        saved_lpb = len(saved_stgl_indices) // num_blocks if saved_stgl_indices else 0
        curr_lpb = len(curr_stgl_indices) // num_blocks if curr_stgl_indices else 0

        # Ánh xạ theo từng Block logic: (block b, layer l) -> (block b, layer l)
        block_mapping = {}
        if saved_lpb > 0 and curr_lpb > 0:
            for b in range(num_blocks):
                for l in range(min(saved_lpb, curr_lpb)):
                    s_idx = b * saved_lpb + l
                    c_idx = b * curr_lpb + l
                    block_mapping[c_idx] = s_idx

        matched_keys = []
        adapted_keys = []
        skipped_keys = []
        self.pretrained_param_names = set()

        for name, curr_param in curr_dict.items():
            if name.startswith("stgl_blocks."):
                parts = name.split(".")
                c_idx = int(parts[1])
                suffix = ".".join(parts[2:])
                if c_idx in block_mapping:
                    saved_name = f"stgl_blocks.{block_mapping[c_idx]}.{suffix}"
                else:
                    saved_name = None
            else:
                saved_name = name

            if saved_name is not None and saved_name in saved_dict:
                saved_param = saved_dict[saved_name]
                if curr_param.shape == saved_param.shape:
                    curr_param.copy_(saved_param)
                    matched_keys.append(f"{name} <- {saved_name}")
                    self.pretrained_param_names.add(name)
                elif (
                        curr_param.ndim == 4
                        and saved_param.ndim == 4
                        and curr_param.shape[:3] == saved_param.shape[:3]
                ):
                    # TCN Conv2d: shape [C_out, C_in, 1, kernel_size]
                    # Khởi tạo 0 toàn bộ kernel mới, sao chép kernel cũ vào các vị trí nhân quả cuối
                    curr_param.zero_()
                    k_saved = saved_param.shape[-1]
                    curr_param[..., -k_saved:].copy_(saved_param)
                    adapted_keys.append(f"{name} (kernel {k_saved} -> {curr_param.shape[-1]})")
                    self.pretrained_param_names.add(name)
                else:
                    skipped_keys.append(f"{name} (shape lệch: {saved_param.shape} vs {curr_param.shape})")
            else:
                # Tầng mới: Zero-init skip_conv; Khởi tạo res_scale = 1.0 (không được zero-init làm ngắt kết nối GCN)
                if name.startswith("stgl_blocks.") and "skip_conv" in name:
                    curr_param.zero_()
                    skipped_keys.append(f"{name} (tầng mới - zero-init)")
                elif name.startswith("stgl_blocks.") and "res_scale" in name:
                    curr_param.fill_(1.0)
                    skipped_keys.append(f"{name} (khởi tạo res_scale = 1.0)")
                else:
                    skipped_keys.append(f"{name} (mới)")

        self.logger.info("=" * 70)
        self.logger.info(f"===> KẾT QUẢ WARM-START TỪ {p.name}:")
        self.logger.info(f"===> Khớp hoàn toàn: {len(matched_keys)} tensors")
        self.logger.info(f"===> Tương thích thích nghi (TCN kernel adaptation): {len(adapted_keys)} tensors")
        self.logger.info(f"===> Khởi tạo mới / Zero-init: {len(skipped_keys)} tensors")
        self.logger.info("=" * 70)

    def _build_scheduler(self) -> Any:
        """Thiết lập lịch trình học Cosine Annealing kèm Linear Warmup."""
        warmup_epochs = max(0, self.cfg.warmup_epochs)
        total_epochs = max(1, self.cfg.epochs)

        def lr_lambda(epoch: int) -> float:
            if epoch < warmup_epochs:
                return float(epoch + 1) / float(max(1, warmup_epochs))
            progress = float(epoch - warmup_epochs) / float(max(1, total_epochs - warmup_epochs))
            return 0.5 * (1.0 + math.cos(math.pi * progress))

        import math
        return torch.optim.lr_scheduler.LambdaLR(self.optimizer, lr_lambda=lr_lambda)

    def _train_epoch(self, epoch: int) -> Dict[str, float]:
        """Thực thi một epoch huấn luyện."""
        self.model.train()

        # Quản lý đóng/mở khóa backbone MobileNetV3 theo epoch
        if epoch <= self.cfg.freeze_backbone_epochs:
            self.model.set_backbone_trainable(False)
        else:
            self.model.set_backbone_trainable(True, unfreeze_last_n_blocks=3)

        total_loss = 0.0
        all_targets: List[int] = []
        all_preds: List[int] = []
        num_batches = len(self.train_loader)

        start_time = time.time()
        pbar = tqdm(
            self.train_loader,
            desc=f"Epoch {epoch:02d}/{self.cfg.epochs:02d} [Train]",
            dynamic_ncols=True,
            leave=False,
        )

        for batch_idx, batch in enumerate(pbar):
            # Nhận dạng các thành phần của batch từ collate_litefat
            if len(batch) == 4:
                batch_frames, batch_landmarks, batch_labels, batch_lengths = batch
            elif len(batch) >= 5:
                batch_frames, batch_landmarks, batch_labels, batch_lengths = batch[:4]
            else:
                batch_frames, batch_landmarks, batch_labels = batch[:3]
                batch_lengths = None

            batch_landmarks = batch_landmarks.to(self.device, non_blocking=True)
            batch_labels = batch_labels.to(self.device, non_blocking=True)
            if batch_frames is not None:
                batch_frames = batch_frames.to(self.device, non_blocking=True)
            if batch_lengths is not None:
                batch_lengths = batch_lengths.to(self.device, non_blocking=True)

            self.optimizer.zero_grad(set_to_none=True)

            # Chạy Forward Pass với AMP
            with torch.amp.autocast(device_type=self.device.type, enabled=self.scaler.is_enabled()):
                frame_logits, clip_logits = self.model(
                    video_frames=batch_frames,
                    landmarks=batch_landmarks,
                    lengths=batch_lengths,
                    return_clip_logits=True,
                )
                loss = self.criterion(
                    frame_logits=frame_logits,
                    targets=batch_labels,
                    clip_logits=clip_logits,
                    lengths=batch_lengths,
                )

            # Backward Pass với GradScaler (kèm kiểm tra an toàn số học)
            if not torch.isfinite(loss):
                self.logger.warning(
                    f"Epoch {epoch:02d} | Batch {batch_idx + 1}/{num_batches}: Loss không hữu hạn ({loss.item()})! "
                    "Bỏ qua cập nhật gradient để bảo vệ trọng số mô hình."
                )
                self.optimizer.zero_grad(set_to_none=True)
                continue

            self.scaler.scale(loss).backward()

            if self.cfg.grad_clip > 0:
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=self.cfg.grad_clip)

            self.scaler.step(self.optimizer)
            self.scaler.update()

            total_loss += loss.item()
            preds = torch.argmax(clip_logits, dim=-1).detach().cpu().numpy()
            targets = batch_labels.detach().cpu().numpy()
            all_preds.extend(preds.tolist())
            all_targets.extend(targets.tolist())

            current_lr = self.optimizer.param_groups[0]["lr"]
            pbar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "avg_loss": f"{total_loss / (batch_idx + 1):.4f}",
                "lr": f"{current_lr:.2e}",
            })

        elapsed = time.time() - start_time
        metrics = compute_metrics(np.array(all_targets), np.array(all_preds))
        metrics["loss"] = total_loss / max(1, num_batches)
        metrics["time"] = elapsed

        return metrics

    @torch.no_grad()
    def _evaluate_epoch(
            self,
            dataloader: Any,
            split_name: str = "val",
            epoch: Optional[int] = None,
            return_raw: bool = False,
    ) -> Any:
        """Đánh giá mô hình trên tập validation hoặc test."""
        self.model.eval()
        total_loss = 0.0
        all_targets: List[int] = []
        all_preds: List[int] = []
        all_probs: List[float] = []
        num_batches = len(dataloader)

        start_time = time.time()
        desc = (
            f"Epoch {epoch:02d}/{self.cfg.epochs:02d} [{split_name.capitalize()}]"
            if epoch is not None
            else f"[{split_name.capitalize()}]"
        )
        pbar = tqdm(
            dataloader,
            desc=desc,
            dynamic_ncols=True,
            leave=False,
        )

        for batch_idx, batch in enumerate(pbar):
            if len(batch) == 4:
                batch_frames, batch_landmarks, batch_labels, batch_lengths = batch
            elif len(batch) >= 5:
                batch_frames, batch_landmarks, batch_labels, batch_lengths = batch[:4]
            else:
                batch_frames, batch_landmarks, batch_labels = batch[:3]
                batch_lengths = None

            batch_landmarks = batch_landmarks.to(self.device, non_blocking=True)
            batch_labels = batch_labels.to(self.device, non_blocking=True)
            if batch_frames is not None:
                batch_frames = batch_frames.to(self.device, non_blocking=True)
            if batch_lengths is not None:
                batch_lengths = batch_lengths.to(self.device, non_blocking=True)

            with torch.amp.autocast(device_type=self.device.type, enabled=self.scaler.is_enabled()):
                frame_logits, clip_logits = self.model(
                    video_frames=batch_frames,
                    landmarks=batch_landmarks,
                    lengths=batch_lengths,
                    return_clip_logits=True,
                )
                loss = self.criterion(
                    frame_logits=frame_logits,
                    targets=batch_labels,
                    clip_logits=clip_logits,
                    lengths=batch_lengths,
                )

            total_loss += loss.item()
            probs = torch.softmax(clip_logits, dim=-1)[:, 1].cpu().numpy()
            preds = torch.argmax(clip_logits, dim=-1).cpu().numpy()
            targets = batch_labels.cpu().numpy()
            all_preds.extend(preds.tolist())
            all_targets.extend(targets.tolist())
            all_probs.extend(probs.tolist())

            pbar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "avg_loss": f"{total_loss / (batch_idx + 1):.4f}",
            })

        elapsed = time.time() - start_time
        targets_np = np.array(all_targets)
        preds_np = np.array(all_preds)
        probs_np = np.array(all_probs)

        metrics = compute_metrics(targets_np, preds_np, probs_np)
        metrics["loss"] = total_loss / max(1, num_batches)
        metrics["time"] = elapsed

        # Quét ngưỡng tối ưu (Threshold Calibration) cân bằng Recall & Specificity
        best_th = 0.50
        candidates = []
        for th in np.linspace(0.35, 0.65, 31):
            m_th = compute_metrics(targets_np, (probs_np >= th).astype(int), probs_np)
            candidates.append((float(th), m_th))

        valid_candidates = [c for c in candidates if c[1]["specificity"] >= 0.65 and c[1]["recall"] >= 0.55]
        if valid_candidates:
            best_th, best_m = max(valid_candidates, key=lambda c: c[1]["f1"])
        else:
            fallback_candidates = [c for c in candidates if c[1]["specificity"] >= 0.50 and c[1]["recall"] >= 0.50]
            if fallback_candidates:
                best_th, best_m = max(fallback_candidates, key=lambda c: c[1]["f1"])
            else:
                best_th, best_m = max(candidates, key=lambda c: c[1]["f1"])

        metrics["optimal_threshold"] = best_th
        metrics["calibrated_f1"] = best_m["f1"]
        metrics["calibrated_accuracy"] = best_m["accuracy"]
        metrics["calibrated_recall"] = best_m["recall"]
        metrics["calibrated_precision"] = best_m["precision"]
        metrics["calibrated_specificity"] = best_m["specificity"]

        auc = metrics.get("auc", 0.5)
        # Composite validation score cân bằng giữa điểm hiệu chuẩn và điểm ngưỡng mặc định 0.50
        metrics["val_score"] = 0.35 * metrics["calibrated_f1"] + 0.20 * metrics["f1"] + 0.25 * auc + 0.20 * metrics[
            "calibrated_accuracy"]

        if return_raw:
            return metrics, targets_np, probs_np
        return metrics

    def train(self) -> None:
        """Vòng lặp huấn luyện chính qua toàn bộ các epoch."""
        if self.start_epoch > self.cfg.epochs:
            self.logger.warning(
                f"Checkpoint đã hoàn thành Epoch {self.start_epoch - 1}, trong khi cfg.epochs là {self.cfg.epochs}. "
                f"Không có epoch mới để huấn luyện. Vui lòng tăng 'epochs' trong TrainConfig "
                f"(ví dụ: epochs={self.start_epoch + 4}) để tiếp tục."
            )
            self.evaluate_test()
            self.writer.close()
            return

        self.logger.info("\n>>> BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN LITEFAT <<<")
        if self.start_epoch > 1:
            self.logger.info(f"Tiếp tục huấn luyện từ Epoch {self.start_epoch:02d} đến {self.cfg.epochs:02d}")

        for epoch in range(self.start_epoch, self.cfg.epochs + 1):
            train_metrics = self._train_epoch(epoch)
            val_metrics = self._evaluate_epoch(self.val_loader, split_name="val", epoch=epoch)

            self.scheduler.step()
            current_lr = self.optimizer.param_groups[0]["lr"]

            # Log chỉ số ra Console
            self.logger.info(
                f"Epoch [{epoch:02d}/{self.cfg.epochs:02d}] \n"
                f"\t Train: Loss={train_metrics['loss']:.4f} | Acc={train_metrics['accuracy'] * 100:.2f}% | F1={train_metrics['f1']:.4f} |\n"
                f"\t Val  : Loss={val_metrics['loss']:.4f} | Acc={val_metrics['accuracy'] * 100:.2f}% | AUC={val_metrics.get('auc', 0.5):.4f} | Score={val_metrics.get('val_score', 0.0):.4f} |\n"
                f"\t Val @0.50: F1={val_metrics['f1']:.4f} (Prec={val_metrics['precision'] * 100:.1f}%, Rec={val_metrics['recall'] * 100:.1f}%) |\n"
                f"\t Val @{val_metrics.get('optimal_threshold', 0.50):.2f}: F1={val_metrics.get('calibrated_f1', 0.0):.4f} (Prec={val_metrics.get('calibrated_precision', 0.0) * 100:.1f}%, Rec={val_metrics.get('calibrated_recall', 0.0) * 100:.1f}%, Spec={val_metrics.get('calibrated_specificity', 0.0) * 100:.1f}%) |\n"
                f"\t LR: {current_lr:.2e} | Time: {train_metrics['time']:.1f}s"
            )

            # Ghi chỉ số vào TensorBoard
            self.writer.add_scalar("Loss/Train", train_metrics["loss"], epoch)
            self.writer.add_scalar("Loss/Val", val_metrics["loss"], epoch)
            self.writer.add_scalar("Accuracy/Train", train_metrics["accuracy"], epoch)
            self.writer.add_scalar("Accuracy/Val", val_metrics["accuracy"], epoch)
            self.writer.add_scalar("F1/Train", train_metrics["f1"], epoch)
            self.writer.add_scalar("F1/Val", val_metrics["f1"], epoch)
            self.writer.add_scalar("AUC/Val", val_metrics.get("auc", 0.5), epoch)
            self.writer.add_scalar("Score/Val", val_metrics.get("val_score", 0.0), epoch)
            self.writer.add_scalar("Calibrated_F1/Val", val_metrics.get("calibrated_f1", 0.0), epoch)
            self.writer.add_scalar("Optimal_Threshold/Val", val_metrics.get("optimal_threshold", 0.50), epoch)
            self.writer.add_scalar("Recall/Val", val_metrics["recall"], epoch)
            self.writer.add_scalar("LR", current_lr, epoch)

            record = {
                "epoch": epoch,
                "train": train_metrics,
                "val": val_metrics,
                "lr": current_lr,
            }
            self.history.append(record)

            # 1. Cập nhật kỷ lục best.pt theo composite val_score (tránh chọn các epoch FP ảo)
            cur_val_score = val_metrics.get("val_score", val_metrics["f1"])
            is_new_best = cur_val_score > self.best_val_score
            if is_new_best:
                self.best_val_score = cur_val_score
                self.best_val_f1 = val_metrics.get("calibrated_f1", val_metrics["f1"])
                self.best_epoch = epoch

            # 2. Lưu checkpoint cuối cùng (last.pt)
            checkpoint_payload = {
                "epoch": epoch,
                "model_state_dict": self.model.state_dict(),
                "optimizer_state_dict": self.optimizer.state_dict(),
                "scheduler_state_dict": self.scheduler.state_dict(),
                "scaler_state_dict": self.scaler.state_dict(),
                "best_epoch": self.best_epoch,
                "best_val_score": self.best_val_score,
                "best_val_f1": self.best_val_f1,
                "val_metrics": val_metrics,
                "train_metrics": train_metrics,
                "config": self.cfg.to_dict(),
            }
            atomic_save(checkpoint_payload, self.save_dir / "last.pt")

            # 3. Lưu checkpoint định kỳ theo save_freq
            if epoch % self.cfg.save_freq == 0:
                atomic_save(checkpoint_payload, self.save_dir / f"epoch_{epoch:02d}.pt")

            # 4. Lưu checkpoint tốt nhất (best.pt)
            if is_new_best:
                atomic_save(checkpoint_payload, self.save_dir / "best.pt")
                with open(self.save_dir / "best_metrics.json", "w", encoding="utf-8") as f:
                    json.dump({"best_epoch": epoch, "metrics": val_metrics}, f, indent=2)
                self.logger.info(
                    f"===> ĐÃ LƯU CHECKPOINT TỐT NHẤT TẠI EPOCH {epoch} "
                    f"(Score: {self.best_val_score:.4f} | Calib F1: {self.best_val_f1:.4f} | AUC: {val_metrics.get('auc', 0.5):.4f})"
                )

        self.logger.info("\n>>> HOÀN THÀNH HUẤN LUYỆN! <<<")
        self.logger.info(f"Checkpoint tốt nhất: Epoch {self.best_epoch} (Val F1: {self.best_val_f1:.4f})")

        # Đánh giá tự động trên tập Test
        self.evaluate_test()
        self.writer.close()

    def evaluate_test(self) -> Dict[str, float]:
        """Tải checkpoint best.pt và đánh giá trên tập Test độc lập kèm Calibration ngưỡng."""
        best_pt_path = self.save_dir / "best.pt"
        if not best_pt_path.is_file():
            self.logger.warning(f"Không tìm thấy checkpoint {best_pt_path} để đánh giá test.")
            return {}

        self.logger.info(f"\n--- ĐÁNH GIÁ TRÊN TẬP TEST VỚI CHECKPOINT TỐT NHẤT (Epoch {self.best_epoch}) ---")
        checkpoint = torch.load(best_pt_path, map_location=self.device, weights_only=False)
        self.model.load_state_dict(checkpoint["model_state_dict"])

        # 1. Tìm ngưỡng tối ưu trên tập Validation
        val_metrics, val_targets, val_probs = self._evaluate_epoch(self.val_loader, split_name="val", return_raw=True)
        best_th = float(val_metrics.get("optimal_threshold", 0.50))
        best_val_f1 = float(val_metrics.get("calibrated_f1", val_metrics["f1"]))

        self.logger.info(
            f"Ngưỡng tối ưu trên Validation: {best_th:.2f} "
            f"(Val Calib F1: {best_val_f1:.4f} vs thr 0.50: {val_metrics['f1']:.4f} | AUC: {val_metrics.get('auc', 0.5):.4f})"
        )

        # 2. Đánh giá trên tập Test với cả ngưỡng mặc định và ngưỡng tối ưu
        test_metrics, test_targets, test_probs = self._evaluate_epoch(self.test_loader, split_name="test",
                                                                      return_raw=True)
        self.logger.info(
            f"[Ngưỡng mặc định 0.50] Test Loss: {test_metrics['loss']:.4f} | "
            f"Test Acc: {test_metrics['accuracy'] * 100:.2f}% | "
            f"Test AUC: {test_metrics.get('auc', 0.5):.4f} | "
            f"Test Precision: {test_metrics['precision'] * 100:.2f}% | "
            f"Test Recall: {test_metrics['recall'] * 100:.2f}% | "
            f"Test Specificity: {test_metrics.get('specificity', 0.0) * 100:.2f}% | "
            f"Test F1: {test_metrics['f1']:.4f} | "
            f"Test F2: {test_metrics['f2']:.4f}"
        )

        calibrated_test_metrics = compute_metrics(test_targets, (test_probs >= best_th).astype(int), test_probs)
        calibrated_test_metrics["loss"] = test_metrics["loss"]
        calibrated_test_metrics["optimal_threshold"] = best_th

        self.logger.info(
            f"[Ngưỡng tối ưu {best_th:.2f}] Test Acc: {calibrated_test_metrics['accuracy'] * 100:.2f}% | "
            f"Test AUC: {calibrated_test_metrics.get('auc', 0.5):.4f} | "
            f"Test Precision: {calibrated_test_metrics['precision'] * 100:.2f}% | "
            f"Test Recall: {calibrated_test_metrics['recall'] * 100:.2f}% | "
            f"Test Specificity: {calibrated_test_metrics.get('specificity', 0.0) * 100:.2f}% | "
            f"Test F1: {calibrated_test_metrics['f1']:.4f} | "
            f"Test F2: {calibrated_test_metrics['f2']:.4f}"
        )

        output_payload = {
            "default_metrics": test_metrics,
            "calibrated_metrics": calibrated_test_metrics,
            "optimal_threshold": best_th,
            "best_epoch": self.best_epoch,
        }
        with open(self.save_dir / "test_metrics.json", "w", encoding="utf-8") as f:
            json.dump(output_payload, f, indent=2)

        return test_metrics

In [12]:
# =====================================================================
# Cell 7: Thực thi Pipeline Huấn luyện LiteFat v5
# =====================================================================
config = TrainConfig()

# Tùy chỉnh tham số cấu hình nếu cần (hoặc giữ nguyên mặc định từ config.py):
# config.use_visual_context = False  # True: Multimodal (Landmark + Face Crops), False: Landmark-Only siêu tốc
# config.batch_size = 8
# config.epochs = 25
# config.resume = "checkpoints/LiteFat_v5/last.pt"

trainer = LiteFatTrainer(config)
trainer.train()


[2026-11-09 00:39:45] [INFO] ======================================================================
[2026-11-09 00:39:45] [INFO] KHỞI TẠO LITEFAT TRAINER (Thiết bị: cuda)
[2026-11-09 00:39:45] [INFO] Chế độ Bối cảnh Thị giác (use_visual_context): False
[2026-11-09 00:39:45] [INFO] Thời gian lấy mẫu (time_interval): 0.05s | Cửa sổ: 12.0s
[2026-11-09 00:39:45] [INFO] Bộ dữ liệu chọn lọc: Tất cả
[2026-11-09 00:39:45] [INFO] ======================================================================
[2026-11-09 00:39:45] [INFO] Đang khởi tạo DataLoaders...
[2026-11-09 00:39:48] [INFO] [Cache] Đã nạp thành công cache train.pt với 5206 mẫu.
[2026-11-09 00:39:48] [INFO] [Cache] Tự động khởi tạo mẫu từ file cache .pt (selected_datasets=None).
[2026-11-09 00:39:48] [INFO] [Cache] Số mẫu sau lọc dataset: 5206
[2026-11-09 00:39:48] [INFO] [Cache] Đã nạp thành công cache val.pt với 1090 mẫu.
[2026-11-09 00:39:48] [INFO] [Cache] Tự động khởi tạo mẫu từ file cache .pt (selected_datasets=None).
[2026-11-0

[2026-11-09 00:43:02] [INFO] Epoch [26/30] 
	 Train: Loss=0.6079 | Acc=81.29% | F1=0.8187 |
	 Val  : Loss=0.8613 | Acc=72.39% | AUC=0.7439 | Score=0.6898 |
	 Val @0.50: F1=0.6223 (Prec=91.2%, Rec=47.2%) |
	 Val @0.35: F1=0.6644 (Prec=84.7%, Rec=54.7%, Spec=90.8%) |
	 LR: 2.71e-04 | Time: 175.7s


[2026-11-09 00:46:15] [INFO] Epoch [27/30] 
	 Train: Loss=0.5968 | Acc=82.29% | F1=0.8245 |
	 Val  : Loss=1.2296 | Acc=56.24% | AUC=0.6657 | Score=0.6249 |
	 Val @0.50: F1=0.6205 (Prec=53.3%, Rec=74.3%) |
	 Val @0.35: F1=0.6371 (Prec=52.6%, Rec=80.8%, Spec=32.4%) |
	 LR: 1.96e-04 | Time: 175.4s


[2026-11-09 00:49:28] [INFO] Epoch [28/30] 
	 Train: Loss=0.6015 | Acc=81.85% | F1=0.8174 |
	 Val  : Loss=0.9988 | Acc=59.36% | AUC=0.7364 | Score=0.6628 |
	 Val @0.50: F1=0.6558 (Prec=55.4%, Rec=80.4%) |
	 Val @0.35: F1=0.6642 (Prec=53.6%, Rec=87.2%, Spec=29.9%) |
	 LR: 1.04e-04 | Time: 175.3s


[2026-11-09 00:52:41] [INFO] Epoch [29/30] 
	 Train: Loss=0.5689 | Acc=84.19% | F1=0.8437 |
	 Val  : Loss=1.0776 | Acc=58.99% | AUC=0.7065 | Score=0.6407 |
	 Val @0.50: F1=0.6272 (Prec=55.8%, Rec=71.6%) |
	 Val @0.56: F1=0.6264 (Prec=56.6%, Rec=70.1%, Spec=50.1%) |
	 LR: 2.86e-05 | Time: 175.5s


[2026-11-09 00:55:54] [INFO] Epoch [30/30] 
	 Train: Loss=0.5563 | Acc=84.56% | F1=0.8497 |
	 Val  : Loss=0.9723 | Acc=62.39% | AUC=0.7329 | Score=0.6678 |
	 Val @0.50: F1=0.6472 (Prec=59.0%, Rec=71.6%) |
	 Val @0.59: F1=0.6487 (Prec=61.3%, Rec=69.0%, Spec=59.5%) |
	 LR: 0.00e+00 | Time: 175.3s


[2026-11-09 00:55:54] [INFO] 
>>> HOÀN THÀNH HUẤN LUYỆN! <<<
[2026-11-09 00:55:54] [INFO] Checkpoint tốt nhất: Epoch 11 (Val F1: 0.7280)
[2026-11-09 00:55:54] [INFO] 
--- ĐÁNH GIÁ TRÊN TẬP TEST VỚI CHECKPOINT TỐT NHẤT (Epoch 11) ---


[2026-11-09 00:56:12] [INFO] Ngưỡng tối ưu trên Validation: 0.63 (Val Calib F1: 0.7269 vs thr 0.50: 0.7225 | AUC: 0.7896)


[2026-11-09 00:56:28] [INFO] [Ngưỡng mặc định 0.50] Test Loss: 0.9549 | Test Acc: 65.03% | Test AUC: 0.6905 | Test Precision: 62.22% | Test Recall: 72.95% | Test Specificity: 57.42% | Test F1: 0.6716 | Test F2: 0.7052
[2026-11-09 00:56:28] [INFO] [Ngưỡng tối ưu 0.63] Test Acc: 64.15% | Test AUC: 0.6905 | Test Precision: 63.19% | Test Recall: 64.33% | Test Specificity: 63.97% | Test F1: 0.6375 | Test F2: 0.6410


In [13]:
!zip -r litefat_results.zip checkpoints 

  adding: checkpoints/ (stored 0%)
  adding: checkpoints/LiteFat_v5/ (stored 0%)
  adding: checkpoints/LiteFat_v5/epoch_29.pt (deflated 9%)
  adding: checkpoints/LiteFat_v5/epoch_21.pt (deflated 9%)
  adding: checkpoints/LiteFat_v5/epoch_27.pt (deflated 9%)
  adding: checkpoints/LiteFat_v5/epoch_06.pt (deflated 9%)
  adding: checkpoints/LiteFat_v5/best_metrics.json (deflated 53%)
  adding: checkpoints/LiteFat_v5/epoch_22.pt (deflated 9%)
  adding: checkpoints/LiteFat_v5/epoch_04.pt (deflated 9%)
  adding: checkpoints/LiteFat_v5/epoch_05.pt (deflated 9%)
  adding: checkpoints/LiteFat_v5/logs/ (stored 0%)
  adding: checkpoints/LiteFat_v5/logs/events.out.tfevents.1794184785.3127c2e07299.4862.1 (deflated 59%)
  adding: checkpoints/LiteFat_v5/logs/events.out.tfevents.1794179866.3127c2e07299.4862.0 (deflated 67%)
  adding: checkpoints/LiteFat_v5/epoch_17.pt (deflated 9%)
  adding: checkpoints/LiteFat_v5/epoch_14.pt (deflated 9%)
  adding: checkpoints/LiteFat_v5/epoch_20.pt (deflated 9%)
  ad